In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
SQLShield - Corrected Validation Pipeline
=========================================

This script replaces the flawed "concat first, then detect text column" data pipeline.

It performs:

1) Correct source-specific loading:
   A: sajid576/Modified_SQL_Dataset.csv      -> Query, Label
   B: syed.../sqliv2.csv                     -> Sentence, Label

2) Strict cleaning:
   - labels must be numeric 0 or 1
   - text must be non-null and length > 2
   - exact duplicates removed within each source
   - one known conflicting normalized group (#NAME?) is removed automatically
     if any normalized group has both labels

3) Correct merged corpus construction:
   - concatenate normalized source tables
   - remove exact duplicate text across sources
   - create normalized_text = lowercase + whitespace collapse

4) Leakage-resistant GROUP-AWARE 80/10/10 split:
   - all rows sharing the same normalized_text stay in ONE partition only
   - unique normalized groups are stratified by binary label
   - split seed fixed at 42
   - assertions verify zero normalized-group overlap across partitions

5) Classical ML baselines on corrected fixed split:
   - Random Forest
   - XGBoost
   - Logistic Regression
   - LinearSVC
   - TF-IDF word uni/bi, max_features=10000

6) Transformer multi-seed stability:
   - CodeBERT x 5 seeds
   - BERT-base x 5 seeds
   - fixed corrected split
   - same hyperparameters as the current paper
   - per-run metrics + mean ± SD

7) Cross-source novel-pattern generalization:
   - A -> residual B (all normalized overlaps with A removed)
   - B -> residual A (all normalized overlaps with B removed)
   - reports class imbalance explicitly
   - metrics include F1, PR-AUC, balanced accuracy, MCC
   - default: CodeBERT, seed 42
   - can be expanded to 5 seeds / both transformers via config

Outputs:
  /kaggle/working/sqlshield_corrected_validation/
    dataset_audit.csv
    conflict_groups.csv
    split_audit.csv
    classical_results.csv
    transformer_multiseed_runs.csv
    transformer_multiseed_summary.csv
    cross_source_runs.csv
    cross_source_summary.csv
    cross_source_overlap.csv
    predictions/*.csv
    histories/*.csv
    config.json
    results.json
"""

from __future__ import annotations

import gc
import json
import os
import random
import re
import time
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    set_seed as hf_set_seed,
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

A_PATH = Path(
    "/kaggle/input/datasets/sajid576/sql-injection-dataset/"
    "Modified_SQL_Dataset.csv"
)
B_PATH = Path(
    "/kaggle/input/datasets/syedsaqlainhussain/sql-injection-dataset/"
    "sqliv2.csv"
)

A_NAME = "sajid576/sql-injection-dataset"
B_NAME = "syedsaqlainhussain/sql-injection-dataset/sqliv2.csv"

SPLIT_SEED = 42
SEEDS = [7, 21, 42, 84, 126]

MODELS = {
    "CodeBERT": "microsoft/codebert-base",
    "BERT-base": "bert-base-uncased",
}

# Cross-source default is intentionally lighter.
# To run 5 seeds externally too, set CROSS_SEEDS = SEEDS.
# To run BERT-base externally too, set CROSS_MODELS = MODELS.
CROSS_MODELS = {
    "CodeBERT": "microsoft/codebert-base",
}
CROSS_SEEDS = [42]

MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRAD_CLIP = 1.0

TFIDF_MAX_FEATURES = 10000

NUM_WORKERS = 0
PIN_MEMORY = True

OUT = Path("/kaggle/working/sqlshield_corrected_validation")
PRED_DIR = OUT / "predictions"
HIST_DIR = OUT / "histories"
CKPT_DIR = OUT / "checkpoints"

for d in [OUT, PRED_DIR, HIST_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# GENERAL UTILITIES
# ============================================================

def banner(s: str) -> None:
    print("\n" + "=" * 84)
    print(s)
    print("=" * 84)


def set_all_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    hf_set_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def normalize_text(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def read_b_csv(path: Path) -> pd.DataFrame:
    # sqliv2.csv is UTF-16 in the currently mounted Kaggle source.
    return pd.read_csv(path, encoding="utf-16", on_bad_lines="skip")


def strict_clean(
    raw: pd.DataFrame,
    text_col: str,
    label_col: str,
    source_name: str,
) -> pd.DataFrame:
    df = raw[[text_col, label_col]].copy()
    df.columns = ["text", "label"]

    df["text"] = df["text"].astype("string").str.strip()
    df["label"] = pd.to_numeric(df["label"], errors="coerce")

    df = df.dropna(subset=["text", "label"])
    df = df[df["label"].isin([0, 1])]
    df = df[df["text"].str.len() > 2].copy()

    df["label"] = df["label"].astype(int)
    df["source"] = source_name

    # Exact source-internal duplicate removal.
    df = df.drop_duplicates(
    subset=["text", "label"],
    keep="first"
).reset_index(drop=True)
    df["normalized_text"] = df["text"].map(normalize_text)

    return df


def load_sources() -> Tuple[pd.DataFrame, pd.DataFrame]:
    if not A_PATH.exists():
        raise FileNotFoundError(f"Dataset A not found: {A_PATH}")
    if not B_PATH.exists():
        raise FileNotFoundError(f"Dataset B not found: {B_PATH}")

    raw_a = pd.read_csv(A_PATH, on_bad_lines="skip")
    raw_b = read_b_csv(B_PATH)

    if "Query" not in raw_a.columns or "Label" not in raw_a.columns:
        raise ValueError(f"Unexpected A columns: {list(raw_a.columns)}")
    if "Sentence" not in raw_b.columns or "Label" not in raw_b.columns:
        raise ValueError(f"Unexpected B columns: {list(raw_b.columns)}")

    a = strict_clean(raw_a, "Query", "Label", A_NAME)
    b = strict_clean(raw_b, "Sentence", "Label", B_NAME)
    return a, b


def source_audit(df: pd.DataFrame, name: str) -> Dict:
    return {
        "dataset": name,
        "rows": int(len(df)),
        "benign": int((df["label"] == 0).sum()),
        "sqli": int((df["label"] == 1).sum()),
        "unique_exact_text": int(df["text"].nunique()),
        "unique_normalized_groups": int(df["normalized_text"].nunique()),
    }


def build_corrected_merged(
    a: pd.DataFrame,
    b: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Merge correctly normalized sources.

    Any normalized_text group with contradictory labels is removed completely.
    Then exact text duplicates across sources are removed.
    """
    combined = pd.concat([a, b], ignore_index=True)

    label_counts = combined.groupby("normalized_text")["label"].nunique()
    conflict_norms = set(label_counts[label_counts > 1].index)

    conflicts = combined[
        combined["normalized_text"].isin(conflict_norms)
    ].copy()

    clean = combined[
        ~combined["normalized_text"].isin(conflict_norms)
    ].copy()

    # Remove exact cross-source duplicates.
    clean = clean.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)

    # Safety: every normalized group must now have exactly one label.
    assert clean.groupby("normalized_text")["label"].nunique().max() == 1

    return clean, conflicts


# ============================================================
# GROUP-AWARE 80/10/10 SPLIT
# ============================================================

def make_group_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    One row per normalized group. Since conflicts were removed, each group has one label.
    """
    g = (
        df.groupby("normalized_text", as_index=False)
          .agg(
              label=("label", "first"),
              n_rows=("text", "size"),
          )
    )
    return g


def group_aware_split(
    df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Stratify UNIQUE NORMALIZED GROUPS by label, then map all group members
    into a single partition.

    This guarantees no normalized_text leakage across train/val/test.
    """
    groups = make_group_table(df)

    train_groups, temp_groups = train_test_split(
        groups,
        test_size=0.20,
        stratify=groups["label"],
        random_state=SPLIT_SEED,
    )

    val_groups, test_groups = train_test_split(
        temp_groups,
        test_size=0.50,
        stratify=temp_groups["label"],
        random_state=SPLIT_SEED,
    )

    train_set = set(train_groups["normalized_text"])
    val_set = set(val_groups["normalized_text"])
    test_set = set(test_groups["normalized_text"])

    assert train_set.isdisjoint(val_set)
    assert train_set.isdisjoint(test_set)
    assert val_set.isdisjoint(test_set)

    train_df = df[df["normalized_text"].isin(train_set)].copy().reset_index(drop=True)
    val_df = df[df["normalized_text"].isin(val_set)].copy().reset_index(drop=True)
    test_df = df[df["normalized_text"].isin(test_set)].copy().reset_index(drop=True)

    # Final zero-leakage assertions.
    assert set(train_df["normalized_text"]).isdisjoint(set(val_df["normalized_text"]))
    assert set(train_df["normalized_text"]).isdisjoint(set(test_df["normalized_text"]))
    assert set(val_df["normalized_text"]).isdisjoint(set(test_df["normalized_text"]))

    return train_df, val_df, test_df


def split_audit_rows(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> List[Dict]:
    rows = []
    total = len(train_df) + len(val_df) + len(test_df)

    for split_name, d in [
        ("train", train_df),
        ("validation", val_df),
        ("test", test_df),
    ]:
        rows.append({
            "split": split_name,
            "rows": int(len(d)),
            "row_pct": float(100 * len(d) / total),
            "benign": int((d["label"] == 0).sum()),
            "sqli": int((d["label"] == 1).sum()),
            "sqli_pct": float(100 * (d["label"] == 1).mean()),
            "normalized_groups": int(d["normalized_text"].nunique()),
        })

    return rows


# ============================================================
# METRICS
# ============================================================

@dataclass
class Metrics:
    accuracy: float
    precision: float
    recall: float
    f1: float
    roc_auc: float
    pr_auc: float
    balanced_accuracy: float
    mcc: float
    tn: int
    fp: int
    fn: int
    tp: int
    errors: int
    n: int


def compute_metrics(y_true, y_pred, y_score) -> Metrics:
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_score = np.asarray(y_score, dtype=float)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    try:
        roc = float(roc_auc_score(y_true, y_score))
    except Exception:
        roc = float("nan")

    try:
        pr = float(average_precision_score(y_true, y_score))
    except Exception:
        pr = float("nan")

    return Metrics(
        accuracy=float(accuracy_score(y_true, y_pred)),
        precision=float(precision_score(y_true, y_pred, zero_division=0)),
        recall=float(recall_score(y_true, y_pred, zero_division=0)),
        f1=float(f1_score(y_true, y_pred, zero_division=0)),
        roc_auc=roc,
        pr_auc=pr,
        balanced_accuracy=float(balanced_accuracy_score(y_true, y_pred)),
        mcc=float(matthews_corrcoef(y_true, y_pred)),
        tn=int(tn),
        fp=int(fp),
        fn=int(fn),
        tp=int(tp),
        errors=int((y_true != y_pred).sum()),
        n=int(len(y_true)),
    )


# ============================================================
# CLASSICAL BASELINES
# ============================================================

def classical_models() -> Dict[str, Pipeline]:
    return {
        "Random Forest": Pipeline([
            ("tfidf", TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                ngram_range=(1, 2),
            )),
            ("clf", RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
            )),
        ]),
        "XGBoost": Pipeline([
            ("tfidf", TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                ngram_range=(1, 2),
            )),
            ("clf", XGBClassifier(
                n_estimators=200,
                random_state=42,
                use_label_encoder=False,
                eval_metric="logloss",
            )),
        ]),
        "Logistic Regression": Pipeline([
            ("tfidf", TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                ngram_range=(1, 2),
            )),
            ("clf", LogisticRegression(
                max_iter=1000,
                random_state=42,
            )),
        ]),
        "SVM": Pipeline([
            ("tfidf", TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                ngram_range=(1, 2),
            )),
            ("clf", LinearSVC(
                max_iter=2000,
                random_state=42,
            )),
        ]),
    }


def run_classical(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> pd.DataFrame:
    banner("CLASSICAL BASELINES ON CORRECTED GROUP-AWARE SPLIT")

    rows = []

    for name, model in classical_models().items():
        print(f"\nTraining {name} ...")
        start = time.time()

        model.fit(train_df["text"], train_df["label"])
        preds = model.predict(test_df["text"])

        clf = model["clf"]
        if hasattr(clf, "predict_proba"):
            scores = model.predict_proba(test_df["text"])[:, 1]
        else:
            scores = model.decision_function(test_df["text"])

        metrics = compute_metrics(test_df["label"], preds, scores)
        row = {
            "model": name,
            **asdict(metrics),
            "elapsed_seconds": time.time() - start,
        }
        rows.append(row)

        pred_df = pd.DataFrame({
            "text": test_df["text"].values,
            "source": test_df["source"].values,
            "true_label": test_df["label"].values,
            "pred_label": preds,
            "score_sqli": scores,
            "correct": (preds == test_df["label"].values).astype(int),
        })
        pred_df.to_csv(
            PRED_DIR / f"classical__{name.replace(' ', '_')}.csv",
            index=False,
        )

        print(
            f"{name}: acc={metrics.accuracy:.6f}, "
            f"f1={metrics.f1:.6f}, roc_auc={metrics.roc_auc:.6f}, "
            f"errors={metrics.errors}, FP={metrics.fp}, FN={metrics.fn}"
        )

    out = pd.DataFrame(rows)
    out.to_csv(OUT / "classical_results.csv", index=False)
    return out


# ============================================================
# TRANSFORMER TRAINING
# ============================================================

class SQLDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer):
        self.texts = df["text"].tolist()
        self.labels = df["label"].astype(int).tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def make_loader(
    df: pd.DataFrame,
    tokenizer,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    gen = torch.Generator()
    gen.manual_seed(seed)

    return DataLoader(
        SQLDataset(df, tokenizer),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY and torch.cuda.is_available(),
        generator=gen if shuffle else None,
    )


def evaluate_transformer(model, loader):
    model.eval()

    preds_all, labels_all, probs_all = [], [], []

    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            labels = batch["labels"].to(DEVICE, non_blocking=True)

            out = model(input_ids=ids, attention_mask=mask)
            probs = torch.softmax(out.logits, dim=-1)[:, 1]
            preds = out.logits.argmax(dim=-1)

            preds_all.extend(preds.cpu().numpy().tolist())
            labels_all.extend(labels.cpu().numpy().tolist())
            probs_all.extend(probs.cpu().numpy().tolist())

    metrics = compute_metrics(labels_all, preds_all, probs_all)

    return (
        metrics,
        np.asarray(preds_all),
        np.asarray(labels_all),
        np.asarray(probs_all),
    )


def train_epoch(model, loader, optimizer, scheduler) -> float:
    model.train()
    total_loss = 0.0
    total_n = 0

    for batch in loader:
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        out = model(
            input_ids=ids,
            attention_mask=mask,
            labels=labels,
        )

        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        bs = labels.size(0)
        total_loss += float(out.loss.item()) * bs
        total_n += bs

    return total_loss / max(total_n, 1)


def train_transformer_once(
    experiment: str,
    model_name: str,
    model_id: str,
    seed: int,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> Dict:
    banner(
        f"{experiment} | {model_name} | seed={seed} | "
        f"train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}"
    )

    set_all_seeds(seed)

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    train_loader = make_loader(train_df, tokenizer, True, seed)
    val_loader = make_loader(val_df, tokenizer, False, seed)
    test_loader = make_loader(test_df, tokenizer, False, seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        id2label={0: "benign", 1: "sqli"},
        label2id={"benign": 0, "sqli": 1},
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps,
    )

    safe_exp = re.sub(r"[^A-Za-z0-9_.-]+", "_", experiment)
    safe_model = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_name)
    ckpt = CKPT_DIR / f"{safe_exp}__{safe_model}__seed{seed}.pt"

    best_val_f1 = -1.0
    best_epoch = None
    history = []
    start = time.time()

    for epoch in range(1, EPOCHS + 1):
        loss = train_epoch(model, train_loader, optimizer, scheduler)
        vm, _, _, _ = evaluate_transformer(model, val_loader)

        history.append({
            "epoch": epoch,
            "train_loss": loss,
            "val_accuracy": vm.accuracy,
            "val_precision": vm.precision,
            "val_recall": vm.recall,
            "val_f1": vm.f1,
            "val_roc_auc": vm.roc_auc,
            "val_pr_auc": vm.pr_auc,
        })

        print(
            f"Epoch {epoch}/{EPOCHS}: loss={loss:.6f}, "
            f"val_f1={vm.f1:.6f}, val_acc={vm.accuracy:.6f}"
        )

        if vm.f1 > best_val_f1:
            best_val_f1 = vm.f1
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt)

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    tm, preds, labels, probs = evaluate_transformer(model, test_loader)

    elapsed = time.time() - start

    pred_df = pd.DataFrame({
        "text": test_df["text"].values,
        "source": test_df["source"].values,
        "true_label": labels,
        "pred_label": preds,
        "prob_sqli": probs,
        "correct": (preds == labels).astype(int),
    })
    pred_df.to_csv(
        PRED_DIR / f"{safe_exp}__{safe_model}__seed{seed}.csv",
        index=False,
    )

    pd.DataFrame(history).to_csv(
        HIST_DIR / f"{safe_exp}__{safe_model}__seed{seed}.csv",
        index=False,
    )

    result = {
        "experiment": experiment,
        "model": model_name,
        "model_id": model_id,
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        **asdict(tm),
        "elapsed_seconds": elapsed,
        "train_n": int(len(train_df)),
        "val_n": int(len(val_df)),
        "test_n": int(len(test_df)),
        "test_sqli": int((test_df["label"] == 1).sum()),
        "test_benign": int((test_df["label"] == 0).sum()),
    }

    print(
        f"TEST: acc={tm.accuracy:.6f}, f1={tm.f1:.6f}, "
        f"roc_auc={tm.roc_auc:.6f}, pr_auc={tm.pr_auc:.6f}, "
        f"bal_acc={tm.balanced_accuracy:.6f}, mcc={tm.mcc:.6f}, "
        f"errors={tm.errors}, FP={tm.fp}, FN={tm.fn}"
    )

    del model, tokenizer, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


SUMMARY_METRICS = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "mcc",
    "errors",
    "fp",
    "fn",
]


def summarize_runs(df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    rows = []

    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))
        row["n_runs"] = int(len(g))

        for m in SUMMARY_METRICS:
            v = pd.to_numeric(g[m], errors="coerce")
            row[f"{m}_mean"] = float(v.mean())
            row[f"{m}_sd"] = float(v.std(ddof=1)) if len(v) > 1 else float("nan")
            row[f"{m}_min"] = float(v.min())
            row[f"{m}_max"] = float(v.max())

        rows.append(row)

    return pd.DataFrame(rows)


def run_transformer_multiseed(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows = []

    for model_name, model_id in MODELS.items():
        for seed in SEEDS:
            rows.append(
                train_transformer_once(
                    experiment="corrected_group_split_multiseed",
                    model_name=model_name,
                    model_id=model_id,
                    seed=seed,
                    train_df=train_df,
                    val_df=val_df,
                    test_df=test_df,
                )
            )

    runs = pd.DataFrame(rows)
    summary = summarize_runs(runs, ["experiment", "model"])

    runs.to_csv(OUT / "transformer_multiseed_runs.csv", index=False)
    summary.to_csv(OUT / "transformer_multiseed_summary.csv", index=False)

    return runs, summary


# ============================================================
# CROSS-SOURCE NOVEL-PATTERN GENERALIZATION
# ============================================================

def residual_external(
    train_source: pd.DataFrame,
    external_source: pd.DataFrame,
) -> Tuple[pd.DataFrame, Dict]:
    train_norms = set(train_source["normalized_text"])
    mask = external_source["normalized_text"].isin(train_norms)

    ext = external_source.loc[~mask].copy().reset_index(drop=True)

    report = {
        "external_before": int(len(external_source)),
        "removed_normalized_overlap": int(mask.sum()),
        "external_after": int(len(ext)),
        "external_remaining_pct": float(100 * len(ext) / len(external_source)),
        "benign_after": int((ext["label"] == 0).sum()),
        "sqli_after": int((ext["label"] == 1).sum()),
        "sqli_pct_after": float(100 * (ext["label"] == 1).mean()),
    }

    if ext["label"].nunique() < 2:
        raise ValueError("Residual external set has only one class.")

    return ext, report


def source_train_val(source_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Group-aware train/val split inside ONE source.
    """
    groups = make_group_table(source_df)

    tr_g, va_g = train_test_split(
        groups,
        test_size=0.10,
        stratify=groups["label"],
        random_state=SPLIT_SEED,
    )

    tr_set = set(tr_g["normalized_text"])
    va_set = set(va_g["normalized_text"])

    train_df = source_df[
        source_df["normalized_text"].isin(tr_set)
    ].copy().reset_index(drop=True)

    val_df = source_df[
        source_df["normalized_text"].isin(va_set)
    ].copy().reset_index(drop=True)

    assert set(train_df["normalized_text"]).isdisjoint(
        set(val_df["normalized_text"])
    )

    return train_df, val_df


def run_cross_direction(
    train_source: pd.DataFrame,
    external_source: pd.DataFrame,
    train_name: str,
    external_name: str,
) -> Tuple[List[Dict], Dict]:
    ext, overlap = residual_external(train_source, external_source)
    train_df, val_df = source_train_val(train_source)

    experiment = f"cross_source_{train_name}_TO_{external_name}"

    overlap.update({
        "experiment": experiment,
        "train_source": train_name,
        "external_source": external_name,
        "train_source_rows": int(len(train_source)),
        "train_partition_rows": int(len(train_df)),
        "val_partition_rows": int(len(val_df)),
    })

    banner(experiment)
    print(json.dumps(overlap, indent=2))

    rows = []

    for model_name, model_id in CROSS_MODELS.items():
        for seed in CROSS_SEEDS:
            r = train_transformer_once(
                experiment=experiment,
                model_name=model_name,
                model_id=model_id,
                seed=seed,
                train_df=train_df,
                val_df=val_df,
                test_df=ext,
            )
            r.update({
                "train_source": train_name,
                "external_source": external_name,
                **overlap,
            })
            rows.append(r)

    return rows, overlap


def run_cross_source(
    a: pd.DataFrame,
    b: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rows = []
    overlap_rows = []

    ab, ab_rep = run_cross_direction(
        a, b,
        "A_sajid576",
        "B_sqliv2",
    )
    rows.extend(ab)
    overlap_rows.append(ab_rep)

    ba, ba_rep = run_cross_direction(
        b, a,
        "B_sqliv2",
        "A_sajid576",
    )
    rows.extend(ba)
    overlap_rows.append(ba_rep)

    runs = pd.DataFrame(rows)
    overlap_df = pd.DataFrame(overlap_rows)
    summary = summarize_runs(
        runs,
        ["experiment", "model", "train_source", "external_source"],
    )

    runs.to_csv(OUT / "cross_source_runs.csv", index=False)
    overlap_df.to_csv(OUT / "cross_source_overlap.csv", index=False)
    summary.to_csv(OUT / "cross_source_summary.csv", index=False)

    return runs, summary, overlap_df


# ============================================================
# MAIN
# ============================================================

def main():
    banner("SQLShield Corrected Validation Pipeline")

    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU count: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

    # --------------------------------------------------------
    # Load sources correctly
    # --------------------------------------------------------
    a, b = load_sources()

    audit = pd.DataFrame([
        source_audit(a, "A_sajid576"),
        source_audit(b, "B_sqliv2"),
    ])

    # Source overlap audit
    exact_overlap = len(set(a["text"]) & set(b["text"]))
    norm_overlap = len(
        set(a["normalized_text"]) & set(b["normalized_text"])
    )

    banner("SOURCE AUDIT")
    print(audit.to_string(index=False))
    print(f"\nExact text overlap: {exact_overlap:,}")
    print(f"Normalized overlap groups: {norm_overlap:,}")

    # --------------------------------------------------------
    # Corrected merged corpus
    # --------------------------------------------------------
    merged, conflicts = build_corrected_merged(a, b)

    conflicts.to_csv(OUT / "conflict_groups.csv", index=False)

    print(f"\nConflicting normalized groups removed: "
          f"{conflicts['normalized_text'].nunique() if len(conflicts) else 0}")
    print(f"Rows removed due to conflicting normalized labels: {len(conflicts)}")

    merged_audit = {
        "dataset": "corrected_merged",
        "rows": int(len(merged)),
        "benign": int((merged["label"] == 0).sum()),
        "sqli": int((merged["label"] == 1).sum()),
        "unique_exact_text": int(merged["text"].nunique()),
        "unique_normalized_groups": int(merged["normalized_text"].nunique()),
        "exact_source_overlap_before_merge": int(exact_overlap),
        "normalized_source_overlap_before_merge": int(norm_overlap),
        "conflicting_normalized_groups_removed": int(
            conflicts["normalized_text"].nunique() if len(conflicts) else 0
        ),
        "conflicting_rows_removed": int(len(conflicts)),
    }

    audit = pd.concat(
        [audit, pd.DataFrame([merged_audit])],
        ignore_index=True,
    )
    audit.to_csv(OUT / "dataset_audit.csv", index=False)

    banner("CORRECTED MERGED CORPUS")
    print(json.dumps(merged_audit, indent=2))

    # --------------------------------------------------------
    # Group-aware fixed split
    # --------------------------------------------------------
    train_df, val_df, test_df = group_aware_split(merged)

    split_rows = split_audit_rows(train_df, val_df, test_df)
    split_df = pd.DataFrame(split_rows)
    split_df.to_csv(OUT / "split_audit.csv", index=False)

    banner("GROUP-AWARE SPLIT AUDIT")
    print(split_df.to_string(index=False))

    # Explicit zero-overlap report
    overlap_checks = {
        "train_val_norm_overlap": len(
            set(train_df["normalized_text"]) & set(val_df["normalized_text"])
        ),
        "train_test_norm_overlap": len(
            set(train_df["normalized_text"]) & set(test_df["normalized_text"])
        ),
        "val_test_norm_overlap": len(
            set(val_df["normalized_text"]) & set(test_df["normalized_text"])
        ),
    }
    print("\nNormalized-group overlap across partitions:")
    print(json.dumps(overlap_checks, indent=2))

    assert all(v == 0 for v in overlap_checks.values())

    # Save split IDs/text for exact reproducibility.
    train_df.to_csv(OUT / "train_corrected.csv", index=False)
    val_df.to_csv(OUT / "val_corrected.csv", index=False)
    test_df.to_csv(OUT / "test_corrected.csv", index=False)

    # --------------------------------------------------------
    # Config snapshot
    # --------------------------------------------------------
    config = {
        "A_PATH": str(A_PATH),
        "B_PATH": str(B_PATH),
        "A_NAME": A_NAME,
        "B_NAME": B_NAME,
        "split_seed": SPLIT_SEED,
        "training_seeds": SEEDS,
        "models": MODELS,
        "cross_models": CROSS_MODELS,
        "cross_seeds": CROSS_SEEDS,
        "max_len": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "grad_clip": GRAD_CLIP,
        "tfidf_max_features": TFIDF_MAX_FEATURES,
        "normalization": "strip + lowercase + collapse internal whitespace",
        "conflict_policy": "remove normalized groups with contradictory labels",
        "split_policy": (
            "stratified split of unique normalized groups; "
            "all rows from each normalized group remain in one partition"
        ),
        "overlap_checks": overlap_checks,
        "device": str(DEVICE),
    }

    (OUT / "config.json").write_text(
        json.dumps(config, indent=2),
        encoding="utf-8",
    )

    # --------------------------------------------------------
    # Classical baselines
    # --------------------------------------------------------
    classical = run_classical(train_df, test_df)

    # --------------------------------------------------------
    # Transformer 5-seed stability
    # --------------------------------------------------------
    multiseed_runs, multiseed_summary = run_transformer_multiseed(
        train_df, val_df, test_df
    )

    # --------------------------------------------------------
    # Cross-source residual generalization
    # --------------------------------------------------------
    cross_runs, cross_summary, cross_overlap = run_cross_source(a, b)

    # --------------------------------------------------------
    # Consolidated JSON
    # --------------------------------------------------------
    results = {
        "config": config,
        "dataset_audit": audit.to_dict(orient="records"),
        "split_audit": split_rows,
        "classical_results": classical.to_dict(orient="records"),
        "transformer_multiseed_runs": multiseed_runs.to_dict(orient="records"),
        "transformer_multiseed_summary": multiseed_summary.to_dict(orient="records"),
        "cross_source_runs": cross_runs.to_dict(orient="records"),
        "cross_source_summary": cross_summary.to_dict(orient="records"),
        "cross_source_overlap": cross_overlap.to_dict(orient="records"),
    }

    (OUT / "results.json").write_text(
        json.dumps(results, indent=2, default=str),
        encoding="utf-8",
    )

    banner("ALL EXPERIMENTS COMPLETE")
    print("\nTransformer multi-seed summary:")
    print(multiseed_summary.to_string(index=False))

    print("\nCross-source summary:")
    print(cross_summary.to_string(index=False))

    print("\nImportant output files:")
    for name in [
        "dataset_audit.csv",
        "split_audit.csv",
        "classical_results.csv",
        "transformer_multiseed_runs.csv",
        "transformer_multiseed_summary.csv",
        "cross_source_runs.csv",
        "cross_source_summary.csv",
        "cross_source_overlap.csv",
        "results.json",
    ]:
        print(" ", OUT / name)

    print(
        "\nNOTE: The residual cross-source external sets are highly imbalanced "
        "(few SQLi samples after normalized-overlap removal). Interpret PR-AUC, "
        "SQLi recall/F1, balanced accuracy, MCC, FP, and FN alongside accuracy."
    )


#if __name__ == "__main__":
  #  main()

In [ ]:
a, b = load_sources()
merged, conflicts = build_corrected_merged(a, b)
train_df, val_df, test_df = group_aware_split(merged)

print(len(train_df), len(val_df), len(test_df))
print(
    len(set(train_df["normalized_text"]) & set(test_df["normalized_text"]))
)

In [ ]:
print("DEVICE:", DEVICE)

# Load the two datasets correctly
a, b = load_sources()

print("\n=== SOURCE A ===")
print("Rows:", len(a))
print("Benign:", (a["label"] == 0).sum())
print("SQLi:", (a["label"] == 1).sum())
print("Normalized groups:", a["normalized_text"].nunique())

print("\n=== SOURCE B ===")
print("Rows:", len(b))
print("Benign:", (b["label"] == 0).sum())
print("SQLi:", (b["label"] == 1).sum())
print("Normalized groups:", b["normalized_text"].nunique())

# Build corrected merged corpus
merged, conflicts = build_corrected_merged(a, b)

print("\n=== CORRECTED MERGED DATASET ===")
print("Rows:", len(merged))
print("Benign:", (merged["label"] == 0).sum())
print("SQLi:", (merged["label"] == 1).sum())
print("Normalized groups:", merged["normalized_text"].nunique())

print("\n=== CONFLICTS REMOVED ===")
print("Conflict rows:", len(conflicts))
print(
    "Conflict groups:",
    conflicts["normalized_text"].nunique() if len(conflicts) else 0
)

# Group-aware split
train_df, val_df, test_df = group_aware_split(merged)

print("\n=== GROUP-AWARE SPLIT ===")

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        name,
        "| rows =", len(df),
        "| benign =", (df["label"] == 0).sum(),
        "| SQLi =", (df["label"] == 1).sum(),
        "| SQLi % =", round((df["label"] == 1).mean() * 100, 2),
        "| groups =", df["normalized_text"].nunique()
    )

print("\n=== LEAKAGE CHECK ===")

print(
    "Train-Val overlap:",
    len(
        set(train_df["normalized_text"])
        & set(val_df["normalized_text"])
    )
)

print(
    "Train-Test overlap:",
    len(
        set(train_df["normalized_text"])
        & set(test_df["normalized_text"])
    )
)

print(
    "Val-Test overlap:",
    len(
        set(val_df["normalized_text"])
        & set(test_df["normalized_text"])
    )
)

In [ ]:
classical_results = run_classical(
    train_df,
    test_df
)

print("\n=== CLASSICAL RESULTS ===")
print(
    classical_results[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "balanced_accuracy",
            "mcc",
            "errors",
            "fp",
            "fn"
        ]
    ]
    .sort_values("f1", ascending=False)
    .to_string(index=False)
)

In [ ]:
codebert_42 = train_transformer_once(
    experiment="corrected_group_split_pilot",
    model_name="CodeBERT",
    model_id="microsoft/codebert-base",
    seed=42,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
)

print("\n=== CODEBERT SEED 42 ===")

for k, v in codebert_42.items():
    print(f"{k}: {v}")

In [ ]:
bert_42 = train_transformer_once(
    experiment="corrected_group_split_pilot",
    model_name="BERT-base",
    model_id="bert-base-uncased",
    seed=42,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
)

print("\n=== BERT-BASE SEED 42 ===")

for k, v in bert_42.items():
    print(f"{k}: {v}")

In [ ]:
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

print("Folder exists:", SAVE_DIR.exists())

if SAVE_DIR.exists():
    for f in sorted(SAVE_DIR.rglob("*")):
        if f.is_file():
            print(f)

In [ ]:
from pathlib import Path

search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

keywords = [
    "CodeBERT",
    "BERT-base",
    "classical_results",
    "Random_Forest",
    "seed42",
    "mcnemar",
]

found = []

for root in search_roots:
    if root.exists():
        for f in root.rglob("*"):
            if f.is_file():
                name = str(f)
                if any(k.lower() in name.lower() for k in keywords):
                    found.append(name)

print("Found files:", len(found))
for x in found:
    print(x)

In [ ]:
a, b = load_sources()
merged, conflicts = build_corrected_merged(a, b)
train_df, val_df, test_df = group_aware_split(merged)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print(
    "Train-Test overlap:",
    len(
        set(train_df["normalized_text"]) &
        set(test_df["normalized_text"])
    )
)

In [ ]:
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(SAVE_DIR / "train_corrected.csv", index=False)
val_df.to_csv(SAVE_DIR / "val_corrected.csv", index=False)
test_df.to_csv(SAVE_DIR / "test_corrected.csv", index=False)

print("Saved successfully:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

for f in SAVE_DIR.glob("*.csv"):
    print(f)

In [ ]:
import pandas as pd
import json
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")
PRED_DIR = SAVE_DIR / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

# Random Forest only
rf_model = classical_models()["Random Forest"]

print("Training Random Forest...")
rf_model.fit(
    train_df["text"],
    train_df["label"]
)

rf_pred = rf_model.predict(test_df["text"])
rf_score = rf_model.predict_proba(test_df["text"])[:, 1]

rf_metrics = compute_metrics(
    test_df["label"],
    rf_pred,
    rf_score
)

print("\n=== RANDOM FOREST ===")
for k, v in rf_metrics.__dict__.items():
    print(f"{k}: {v}")

# Save predictions
rf_predictions = pd.DataFrame({
    "text": test_df["text"].values,
    "source": test_df["source"].values,
    "true_label": test_df["label"].values,
    "pred_label": rf_pred,
    "score_sqli": rf_score,
    "correct": (
        rf_pred == test_df["label"].values
    ).astype(int)
})

rf_predictions.to_csv(
    PRED_DIR / "classical__Random_Forest.csv",
    index=False
)

# Save metrics
with open(
    SAVE_DIR / "random_forest_result.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        rf_metrics.__dict__,
        f,
        indent=2,
        default=str
    )

print("\nSaved:")
print(PRED_DIR / "classical__Random_Forest.csv")
print(SAVE_DIR / "random_forest_result.json")

In [ ]:
codebert_42 = train_transformer_once(
    experiment="corrected_group_split_pilot",
    model_name="CodeBERT",
    model_id="microsoft/codebert-base",
    seed=42,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
)

print("\n=== CODEBERT SEED 42 ===")

for k, v in codebert_42.items():
    print(f"{k}: {v}")

In [ ]:
import json
import shutil
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

# Save CodeBERT metrics explicitly
with open(
    SAVE_DIR / "codebert_seed42_result.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        codebert_42,
        f,
        indent=2,
        default=str
    )

print("=== CURRENT FILES ===")
for f in sorted(SAVE_DIR.rglob("*")):
    if f.is_file():
        print(f)

# Create full backup ZIP
backup_path = shutil.make_archive(
    "/kaggle/working/sqlshield_after_codebert",
    "zip",
    SAVE_DIR
)

print("\nBackup created:")
print(backup_path)

In [ ]:
from IPython.display import FileLink, display

display(
    FileLink(
        "/kaggle/working/sqlshield_after_codebert.zip"
    )
)

In [ ]:
import os

path = "/kaggle/working/sqlshield_after_codebert.zip"

print("Exists:", os.path.exists(path))
print("Size MB:", round(os.path.getsize(path) / 1024**2, 2))

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<a href="files/sqlshield_after_codebert.zip"
   download
   style="font-size:18px;font-weight:bold;">
   اضغط هنا لتنزيل SQLShield Backup
</a>
"""))

In [ ]:
bert_42 = train_transformer_once(
    experiment="corrected_group_split_pilot",
    model_name="BERT-base",
    model_id="bert-base-uncased",
    seed=42,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
)

print("\n=== BERT-BASE SEED 42 ===")

for k, v in bert_42.items():
    print(f"{k}: {v}")

In [ ]:
import json
import shutil
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

# Save BERT metrics
with open(
    SAVE_DIR / "bert_seed42_result.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        bert_42,
        f,
        indent=2,
        default=str
    )

# Show current files
print("=== CURRENT FILES ===")
for f in sorted(SAVE_DIR.rglob("*")):
    if f.is_file():
        print(f)

# Full backup after BERT
backup_path = shutil.make_archive(
    "/kaggle/working/sqlshield_after_bert",
    "zip",
    SAVE_DIR
)

print("\nBackup created:")
print(backup_path)

In [ ]:
from pathlib import Path
import pandas as pd
from scipy.stats import binomtest

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")
PRED_DIR = SAVE_DIR / "predictions"

# Load predictions
cb = pd.read_csv(
    PRED_DIR / "corrected_group_split_pilot__CodeBERT__seed42.csv"
)

bert = pd.read_csv(
    PRED_DIR / "corrected_group_split_pilot__BERT-base__seed42.csv"
)

rf = pd.read_csv(
    PRED_DIR / "classical__Random_Forest.csv"
)

# Verify all models used exactly the same test set
assert len(cb) == len(bert) == len(rf) == 5654
assert (cb["text"].values == bert["text"].values).all()
assert (cb["text"].values == rf["text"].values).all()

assert (
    cb["true_label"].values ==
    bert["true_label"].values
).all()

assert (
    cb["true_label"].values ==
    rf["true_label"].values
).all()

def exact_mcnemar(df1, df2, name1, name2):

    correct1 = (
        df1["pred_label"].values ==
        df1["true_label"].values
    )

    correct2 = (
        df2["pred_label"].values ==
        df2["true_label"].values
    )

    # Model 1 wrong, Model 2 correct
    b = int((~correct1 & correct2).sum())

    # Model 1 correct, Model 2 wrong
    c = int((correct1 & ~correct2).sum())

    n = b + c

    if n == 0:
        p = 1.0
    else:
        p = binomtest(
            min(b, c),
            n=n,
            p=0.5,
            alternative="two-sided"
        ).pvalue

    return {
        "comparison": f"{name1} vs {name2}",
        "model1_wrong_model2_correct": b,
        "model1_correct_model2_wrong": c,
        "discordant_pairs": n,
        "exact_p_value": p
    }

results = [
    exact_mcnemar(cb, bert, "CodeBERT", "BERT-base"),
    exact_mcnemar(cb, rf, "CodeBERT", "Random Forest"),
    exact_mcnemar(bert, rf, "BERT-base", "Random Forest"),
]

mcnemar_df = pd.DataFrame(results)

print("\n=== EXACT McNEMAR RESULTS ===")
print(mcnemar_df.to_string(index=False))

mcnemar_df.to_csv(
    SAVE_DIR / "pilot_exact_mcnemar.csv",
    index=False
)

print("\nSaved:")
print(SAVE_DIR / "pilot_exact_mcnemar.csv")

In [ ]:
from pathlib import Path
import pandas as pd
import json
import gc
import torch

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

# Reload exact saved splits
train_df = pd.read_csv(SAVE_DIR / "train_corrected.csv")
val_df   = pd.read_csv(SAVE_DIR / "val_corrected.csv")
test_df  = pd.read_csv(SAVE_DIR / "test_corrected.csv")

# Existing seed-42 results
with open(SAVE_DIR / "codebert_seed42_result.json", "r") as f:
    codebert_42_saved = json.load(f)

with open(SAVE_DIR / "bert_seed42_result.json", "r") as f:
    bert_42_saved = json.load(f)

# We already completed seed 42
remaining_seeds = [7, 21, 84, 126]

models_to_run = {
    "CodeBERT": "microsoft/codebert-base",
    "BERT-base": "bert-base-uncased",
}

# Start table with already completed seed 42
all_results = []

cb42 = dict(codebert_42_saved)
cb42["experiment"] = "corrected_group_split_multiseed"
all_results.append(cb42)

b42 = dict(bert_42_saved)
b42["experiment"] = "corrected_group_split_multiseed"
all_results.append(b42)

RESULT_FILE = SAVE_DIR / "transformer_multiseed_runs.csv"

# If a partial results file already exists, use it
if RESULT_FILE.exists():
    old = pd.read_csv(RESULT_FILE)

    existing_keys = set(
        zip(old["model"], old["seed"])
    )

    for row in old.to_dict("records"):
        key = (row["model"], int(row["seed"]))

        if key not in {
            (r["model"], int(r["seed"]))
            for r in all_results
        }:
            all_results.append(row)
else:
    existing_keys = {
        ("CodeBERT", 42),
        ("BERT-base", 42),
    }


for model_name, model_id in models_to_run.items():

    for seed in remaining_seeds:

        key = (model_name, seed)

        # Resume-safe
        completed = {
            (r["model"], int(r["seed"]))
            for r in all_results
        }

        if key in completed:
            print(f"SKIP: {model_name} seed={seed} already completed")
            continue

        print("\n" + "=" * 80)
        print(f"RUNNING: {model_name} | seed={seed}")
        print("=" * 80)

        result = train_transformer_once(
            experiment="corrected_group_split_multiseed",
            model_name=model_name,
            model_id=model_id,
            seed=seed,
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
        )

        all_results.append(result)

        # SAVE IMMEDIATELY AFTER EACH SEED
        runs_df = pd.DataFrame(all_results)

        runs_df.to_csv(
            RESULT_FILE,
            index=False
        )

        # Separate JSON backup
        with open(
            SAVE_DIR / "transformer_multiseed_runs.json",
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                all_results,
                f,
                indent=2,
                default=str
            )

        print("\nSAVED AFTER THIS SEED:")
        print(RESULT_FILE)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# FINAL SUMMARY
# ============================================================

runs_df = pd.DataFrame(all_results)

metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "mcc",
    "errors",
    "fp",
    "fn",
]

summary_rows = []

for model_name, group in runs_df.groupby("model"):

    row = {
        "model": model_name,
        "n_seeds": len(group)
    }

    for metric in metrics:
        values = pd.to_numeric(
            group[metric],
            errors="coerce"
        )

        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_sd"] = values.std(ddof=1)
        row[f"{metric}_min"] = values.min()
        row[f"{metric}_max"] = values.max()

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    SAVE_DIR / "transformer_multiseed_summary.csv",
    index=False
)

print("\n=== MULTI-SEED RUNS ===")
print(
    runs_df[
        ["model", "seed", "best_epoch",
         "accuracy", "f1", "errors", "fp", "fn"]
    ]
    .sort_values(["model", "seed"])
    .to_string(index=False)
)

print("\n=== MULTI-SEED SUMMARY ===")
print(summary_df.to_string(index=False))

In [ ]:
print("session started")

In [ ]:
from pathlib import Path
import pandas as pd

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")
RESULT_FILE = SAVE_DIR / "transformer_multiseed_runs.csv"

print("SAVE_DIR exists:", SAVE_DIR.exists())
print("Results file exists:", RESULT_FILE.exists())

if RESULT_FILE.exists():
    runs = pd.read_csv(RESULT_FILE)

    print("\n=== SAVED RUNS ===")
    print(
        runs[
            ["model", "seed", "best_epoch",
             "accuracy", "f1", "errors", "fp", "fn"]
        ]
        .sort_values(["model", "seed"])
        .to_string(index=False)
    )

    print("\n=== COMPLETED SEEDS ===")
    print(
        runs.groupby(["model", "seed"])
            .size()
            .to_string()
    )

In [ ]:
!kaggle kernels output mohammadalkhazaleh/notebook691d0eee69 -p /kaggle/working/version4_restore

In [ ]:
import shutil
from pathlib import Path

SRC = Path(
    "/kaggle/working/version4_restore/sqlshield_corrected_validation"
)

DST = Path(
    "/kaggle/working/sqlshield_corrected_validation"
)

if DST.exists():
    shutil.rmtree(DST)

shutil.copytree(SRC, DST)

print("Restored to:")
print(DST)

print("\nFiles:")
for f in sorted(DST.rglob("*")):
    if f.is_file():
        print(f)

In [ ]:
from pathlib import Path
import pandas as pd
import json

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

train_df = pd.read_csv(SAVE_DIR / "train_corrected.csv")
val_df   = pd.read_csv(SAVE_DIR / "val_corrected.csv")
test_df  = pd.read_csv(SAVE_DIR / "test_corrected.csv")

with open(SAVE_DIR / "codebert_seed42_result.json") as f:
    cb42 = json.load(f)

with open(SAVE_DIR / "bert_seed42_result.json") as f:
    b42 = json.load(f)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("CodeBERT seed42 errors:", cb42["errors"])
print("CodeBERT seed42 F1:", cb42["f1"])

print("BERT seed42 errors:", b42["errors"])
print("BERT seed42 F1:", b42["f1"])

print(
    "train_transformer_once available:",
    "train_transformer_once" in globals()
)

In [ ]:
print(
    "train_transformer_once available:",
    "train_transformer_once" in globals()
)

print("DEVICE:", DEVICE)

In [ ]:
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

expected = []

for model in ["CodeBERT", "BERT-base"]:
    for seed in [7, 21, 42, 84, 126]:

        model_file = model.replace("-", "-")

        if seed == 42:
            pred = SAVE_DIR / "predictions" / \
                f"corrected_group_split_pilot__{model_file}__seed42.csv"
            hist = SAVE_DIR / "histories" / \
                f"corrected_group_split_pilot__{model_file}__seed42.csv"
            ckpt = SAVE_DIR / "checkpoints" / \
                f"corrected_group_split_pilot__{model_file}__seed42.pt"
        else:
            pred = SAVE_DIR / "predictions" / \
                f"corrected_group_split_multiseed__{model_file}__seed{seed}.csv"
            hist = SAVE_DIR / "histories" / \
                f"corrected_group_split_multiseed__{model_file}__seed{seed}.csv"
            ckpt = SAVE_DIR / "checkpoints" / \
                f"corrected_group_split_multiseed__{model_file}__seed{seed}.pt"

        expected.append({
            "model": model,
            "seed": seed,
            "prediction": pred.exists(),
            "history": hist.exists(),
            "checkpoint": ckpt.exists()
        })

import pandas as pd

audit = pd.DataFrame(expected)

print("\n=== MULTI-SEED FILE AUDIT ===")
print(audit.to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

train_df = pd.read_csv(SAVE_DIR / "train_corrected.csv")
val_df   = pd.read_csv(SAVE_DIR / "val_corrected.csv")
test_df  = pd.read_csv(SAVE_DIR / "test_corrected.csv")

print("Function available:", "train_transformer_once" in globals())
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("DEVICE:", DEVICE)

In [ ]:
import json
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

codebert_21 = train_transformer_once(
    experiment="corrected_group_split_multiseed",
    model_name="CodeBERT",
    model_id="microsoft/codebert-base",
    seed=21,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
)

# Save metrics independently
RESULT_PATH = SAVE_DIR / "codebert_seed21_result.json"

with open(
    RESULT_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        codebert_21,
        f,
        indent=2,
        default=str
    )

print("\n=== CODEBERT SEED 21 ===")

for k, v in codebert_21.items():
    print(f"{k}: {v}")

print("\nSaved result:")
print(RESULT_PATH)

print("\nExpected artifacts:")
print(
    SAVE_DIR /
    "predictions/corrected_group_split_multiseed__CodeBERT__seed21.csv"
)
print(
    SAVE_DIR /
    "histories/corrected_group_split_multiseed__CodeBERT__seed21.csv"
)
print(
    SAVE_DIR /
    "checkpoints/corrected_group_split_multiseed__CodeBERT__seed21.pt"
)

In [ ]:
import json
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

bert_126 = train_transformer_once(
    experiment="corrected_group_split_multiseed",
    model_name="BERT-base",
    model_id="bert-base-uncased",
    seed=126,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
)

# Save metrics independently
RESULT_PATH = SAVE_DIR / "bert_seed126_result.json"

with open(
    RESULT_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        bert_126,
        f,
        indent=2,
        default=str
    )

print("\n=== BERT-BASE SEED 126 ===")

for k, v in bert_126.items():
    print(f"{k}: {v}")

print("\nSaved result:")
print(RESULT_PATH)

print("\nExpected artifacts:")
print(
    SAVE_DIR /
    "predictions/corrected_group_split_multiseed__BERT-base__seed126.csv"
)
print(
    SAVE_DIR /
    "histories/corrected_group_split_multiseed__BERT-base__seed126.csv"
)
print(
    SAVE_DIR /
    "checkpoints/corrected_group_split_multiseed__BERT-base__seed126.pt"
)

In [14]:
from pathlib import Path
import pandas as pd
import json
import numpy as np

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

# ============================================================
# 1. VERIFY NEWLY SAVED RESULTS
# ============================================================

required_files = [
    "codebert_seed42_result.json",
    "bert_seed42_result.json",
    "codebert_seed21_result.json",
    "bert_seed126_result.json",
]

print("=== REQUIRED FILES ===")

for name in required_files:
    p = SAVE_DIR / name
    print(name, "->", p.exists())

assert all((SAVE_DIR / x).exists() for x in required_files), \
    "One or more required JSON files are missing."

# ============================================================
# 2. LOAD SAVED RUNS
# ============================================================

def load_json(name):
    with open(SAVE_DIR / name, "r", encoding="utf-8") as f:
        return json.load(f)

cb42  = load_json("codebert_seed42_result.json")
bert42 = load_json("bert_seed42_result.json")
cb21  = load_json("codebert_seed21_result.json")
bert126 = load_json("bert_seed126_result.json")

# ============================================================
# 3. HELPER FOR RECOVERED NOTEBOOK RESULTS
# ============================================================

TEST_N = 5654
TEST_BENIGN = 3450
TEST_SQLI = 2204

def recovered_row(
    model,
    seed,
    best_epoch,
    accuracy,
    f1,
    roc_auc,
    pr_auc,
    balanced_accuracy,
    mcc,
    fp,
    fn
):
    tn = TEST_BENIGN - fp
    tp = TEST_SQLI - fn

    precision = tp / (tp + fp)
    recall = tp / (tp + fn)

    return {
        "experiment": "corrected_group_split_multiseed",
        "model": model,
        "seed": seed,
        "best_epoch": best_epoch,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "balanced_accuracy": balanced_accuracy,
        "mcc": mcc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "errors": fp + fn,
        "n": TEST_N,
        "source": "recovered_from_notebook_output"
    }

# ============================================================
# 4. RESULTS RECOVERED FROM THE ORIGINAL NOTEBOOK OUTPUT
# ============================================================

recovered = [

    # ---------------- CodeBERT seed 7 ----------------
    recovered_row(
        model="CodeBERT",
        seed=7,
        best_epoch=2,
        accuracy=0.999469,
        f1=0.999320,
        roc_auc=0.999997,
        pr_auc=0.999995,
        balanced_accuracy=0.999483,
        mcc=0.998885,
        fp=2,
        fn=1
    ),

    # ---------------- CodeBERT seed 84 ----------------
    recovered_row(
        model="CodeBERT",
        seed=84,
        best_epoch=4,
        accuracy=0.999469,
        f1=0.999319,
        roc_auc=0.999652,
        pr_auc=0.999751,
        balanced_accuracy=0.999401,
        mcc=0.998885,
        fp=1,
        fn=2
    ),

    # ---------------- CodeBERT seed 126 ----------------
    recovered_row(
        model="CodeBERT",
        seed=126,
        best_epoch=3,
        accuracy=0.999293,
        f1=0.999093,
        roc_auc=0.999986,
        pr_auc=0.999979,
        balanced_accuracy=0.999338,
        mcc=0.998513,
        fp=3,
        fn=1
    ),

    # ---------------- BERT seed 7 ----------------
    recovered_row(
        model="BERT-base",
        seed=7,
        best_epoch=3,
        accuracy=0.998408,
        f1=0.997957,
        roc_auc=0.999570,
        pr_auc=0.999674,
        balanced_accuracy=0.998204,
        mcc=0.996654,
        fp=3,
        fn=6
    ),

    # ---------------- BERT seed 21 ----------------
    recovered_row(
        model="BERT-base",
        seed=21,
        best_epoch=3,
        accuracy=0.998585,
        f1=0.998185,
        roc_auc=0.999948,
        pr_auc=0.999925,
        balanced_accuracy=0.998513,
        mcc=0.997026,
        fp=4,
        fn=4
    ),

    # ---------------- BERT seed 84 ----------------
    recovered_row(
        model="BERT-base",
        seed=84,
        best_epoch=3,
        accuracy=0.999293,
        f1=0.999092,
        roc_auc=0.999649,
        pr_auc=0.999663,
        balanced_accuracy=0.999093,
        mcc=0.998513,
        fp=0,
        fn=4
    ),
]

# ============================================================
# 5. NORMALIZE JSON RESULTS
# ============================================================

def saved_row(d):
    keep = {
        "experiment": d.get(
            "experiment",
            "corrected_group_split_multiseed"
        ),
        "model": d["model"],
        "seed": int(d["seed"]),
        "best_epoch": d["best_epoch"],
        "accuracy": float(d["accuracy"]),
        "precision": float(d["precision"]),
        "recall": float(d["recall"]),
        "f1": float(d["f1"]),
        "roc_auc": float(d["roc_auc"]),
        "pr_auc": float(d["pr_auc"]),
        "balanced_accuracy": float(d["balanced_accuracy"]),
        "mcc": float(d["mcc"]),
        "tn": int(d["tn"]),
        "fp": int(d["fp"]),
        "fn": int(d["fn"]),
        "tp": int(d["tp"]),
        "errors": int(d["errors"]),
        "n": int(d["n"]),
        "source": "saved_json"
    }

    keep["experiment"] = "corrected_group_split_multiseed"
    return keep

saved = [
    saved_row(cb21),
    saved_row(cb42),
    saved_row(bert42),
    saved_row(bert126),
]

# ============================================================
# 6. BUILD COMPLETE 10-RUN TABLE
# ============================================================

runs = pd.DataFrame(recovered + saved)

runs = (
    runs
    .sort_values(["model", "seed"])
    .reset_index(drop=True)
)

# Verify 5 seeds per model
expected_seeds = {7, 21, 42, 84, 126}

for model in ["CodeBERT", "BERT-base"]:
    found = set(
        runs.loc[runs["model"] == model, "seed"].astype(int)
    )

    print(model, "seeds:", sorted(found))

    assert found == expected_seeds, \
        f"Missing or unexpected seeds for {model}: {found}"

assert len(runs) == 10

# ============================================================
# 7. MULTI-SEED SUMMARY
# ============================================================

metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "mcc",
    "errors",
    "fp",
    "fn",
]

summary_rows = []

for model, g in runs.groupby("model"):

    row = {
        "model": model,
        "n_seeds": len(g)
    }

    for metric in metrics:
        x = pd.to_numeric(g[metric], errors="coerce")

        row[f"{metric}_mean"] = x.mean()
        row[f"{metric}_sd"] = x.std(ddof=1)
        row[f"{metric}_min"] = x.min()
        row[f"{metric}_max"] = x.max()

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)

# ============================================================
# 8. SAVE FINAL TABLES
# ============================================================

runs.to_csv(
    SAVE_DIR / "transformer_multiseed_runs_FINAL.csv",
    index=False
)

summary.to_csv(
    SAVE_DIR / "transformer_multiseed_summary_FINAL.csv",
    index=False
)

# ============================================================
# 9. DISPLAY
# ============================================================

print("\n=== FINAL 5-SEED RUNS ===")

print(
    runs[
        [
            "model",
            "seed",
            "best_epoch",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "balanced_accuracy",
            "mcc",
            "errors",
            "fp",
            "fn",
            "source"
        ]
    ].to_string(index=False)
)

print("\n=== FINAL 5-SEED SUMMARY ===")

cols = [
    "model",
    "n_seeds",
    "accuracy_mean",
    "accuracy_sd",
    "f1_mean",
    "f1_sd",
    "roc_auc_mean",
    "roc_auc_sd",
    "pr_auc_mean",
    "pr_auc_sd",
    "balanced_accuracy_mean",
    "balanced_accuracy_sd",
    "mcc_mean",
    "mcc_sd",
    "errors_mean",
    "errors_sd",
]

print(
    summary[cols].to_string(index=False)
)

print("\nSaved:")
print(SAVE_DIR / "transformer_multiseed_runs_FINAL.csv")
print(SAVE_DIR / "transformer_multiseed_summary_FINAL.csv")

=== REQUIRED FILES ===
codebert_seed42_result.json -> True
bert_seed42_result.json -> True
codebert_seed21_result.json -> True
bert_seed126_result.json -> True
CodeBERT seeds: [7, 21, 42, 84, 126]
BERT-base seeds: [7, 21, 42, 84, 126]

=== FINAL 5-SEED RUNS ===
    model  seed  best_epoch  accuracy  precision   recall       f1  roc_auc   pr_auc  balanced_accuracy      mcc  errors  fp  fn                         source
BERT-base     7           3  0.998408   0.998637 0.997278 0.997957 0.999570 0.999674           0.998204 0.996654       9   3   6 recovered_from_notebook_output
BERT-base    21           3  0.998585   0.998185 0.998185 0.998185 0.999948 0.999925           0.998513 0.997026       8   4   4 recovered_from_notebook_output
BERT-base    42           3  0.998939   0.999092 0.998185 0.998638 0.999955 0.999932           0.998803 0.997769       6   2   4                     saved_json
BERT-base    84           3  0.999293   1.000000 0.998185 0.999092 0.999649 0.999663           0.9

In [15]:
from pathlib import Path
import pandas as pd
import json

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Load the two sources correctly
a, b = load_sources()

print("A:", len(a), a["label"].value_counts().to_dict())
print("B:", len(b), b["label"].value_counts().to_dict())

# Check residual external sets
b_external, ab_report = residual_external(a, b)
a_external, ba_report = residual_external(b, a)

print("\n=== A -> B ===")
print(ab_report)

print("\n=== B -> A ===")
print(ba_report)

A: 30766 {0: 19481, 1: 11285}
B: 33537 {0: 22171, 1: 11366}

=== A -> B ===
{'external_before': 33537, 'removed_normalized_overlap': 18355, 'external_after': 15182, 'external_remaining_pct': 45.2694039419149, 'benign_after': 15004, 'sqli_after': 178, 'sqli_pct_after': 1.1724410486101964}

=== B -> A ===
{'external_before': 30766, 'removed_normalized_overlap': 18356, 'external_after': 12410, 'external_remaining_pct': 40.336735357212504, 'benign_after': 12314, 'sqli_after': 96, 'sqli_pct_after': 0.7735697018533441}


In [16]:
import pandas as pd

# ============================================================
# CROSS-SOURCE CONFLICT CLEANING
# ============================================================

combined_cross = pd.concat([a, b], ignore_index=True)

# Find normalized groups that have contradictory labels
label_counts = (
    combined_cross
    .groupby("normalized_text")["label"]
    .nunique()
)

conflict_norms = set(
    label_counts[label_counts > 1].index
)

conflict_rows = combined_cross[
    combined_cross["normalized_text"].isin(conflict_norms)
].copy()

print("Conflicting normalized groups:", len(conflict_norms))
print("Conflicting rows:", len(conflict_rows))

print("\n=== CONFLICT ROWS ===")
print(
    conflict_rows[
        ["text", "label", "source", "normalized_text"]
    ].to_string(index=False)
)

# Remove the ENTIRE conflicting normalized group from BOTH sources
a_cross = a[
    ~a["normalized_text"].isin(conflict_norms)
].copy().reset_index(drop=True)

b_cross = b[
    ~b["normalized_text"].isin(conflict_norms)
].copy().reset_index(drop=True)

# Safety checks
assert (
    a_cross.groupby("normalized_text")["label"].nunique().max()
    == 1
)

assert (
    b_cross.groupby("normalized_text")["label"].nunique().max()
    == 1
)

print("\n=== CLEAN CROSS-SOURCE DATA ===")
print(
    "A_cross:",
    len(a_cross),
    a_cross["label"].value_counts().to_dict()
)

print(
    "B_cross:",
    len(b_cross),
    b_cross["label"].value_counts().to_dict()
)

# ============================================================
# RECHECK RESIDUAL EXTERNAL SETS
# ============================================================

b_external, ab_report = residual_external(
    a_cross,
    b_cross
)

a_external, ba_report = residual_external(
    b_cross,
    a_cross
)

print("\n=== CORRECTED A -> B ===")
print(ab_report)

print("\n=== CORRECTED B -> A ===")
print(ba_report)

# Absolute safety:
assert set(a_cross["normalized_text"]).isdisjoint(
    set(b_external["normalized_text"])
)

assert set(b_cross["normalized_text"]).isdisjoint(
    set(a_external["normalized_text"])
)

print("\nZero cross-source normalized overlap: VERIFIED")

Conflicting normalized groups: 1
Conflicting rows: 2

=== CONFLICT ROWS ===
  text  label                         source normalized_text
#NAME?      1 sajid576/sql-injection-dataset          #name?
#NAME?      0 sajid576/sql-injection-dataset          #name?

=== CLEAN CROSS-SOURCE DATA ===
A_cross: 30764 {0: 19480, 1: 11284}
B_cross: 33537 {0: 22171, 1: 11366}

=== CORRECTED A -> B ===
{'external_before': 33537, 'removed_normalized_overlap': 18355, 'external_after': 15182, 'external_remaining_pct': 45.2694039419149, 'benign_after': 15004, 'sqli_after': 178, 'sqli_pct_after': 1.1724410486101964}

=== CORRECTED B -> A ===
{'external_before': 30764, 'removed_normalized_overlap': 18356, 'external_after': 12408, 'external_remaining_pct': 40.33285658561955, 'benign_after': 12313, 'sqli_after': 95, 'sqli_pct_after': 0.7656350741457124}

Zero cross-source normalized overlap: VERIFIED


In [17]:
import pandas as pd
import json
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

# ============================================================
# CROSS-SOURCE: A -> B
# ============================================================

ab_rows, ab_overlap = run_cross_direction(
    train_source=a_cross,
    external_source=b_cross,
    train_name="A_sajid576",
    external_name="B_sqliv2",
)

ab_df = pd.DataFrame(ab_rows)

print("\n=== FINAL A -> B RESULT ===")

cols = [
    "model",
    "seed",
    "best_epoch",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "mcc",
    "tn",
    "fp",
    "fn",
    "tp",
    "errors",
    "test_n",
    "test_sqli",
    "test_benign",
]

print(ab_df[cols].to_string(index=False))

# Save run
ab_df.to_csv(
    SAVE_DIR / "cross_source_A_to_B_FINAL.csv",
    index=False
)

# Save overlap/audit report
with open(
    SAVE_DIR / "cross_source_A_to_B_overlap_FINAL.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(ab_overlap, f, indent=2, default=str)

print("\nSaved:")
print(SAVE_DIR / "cross_source_A_to_B_FINAL.csv")
print(SAVE_DIR / "cross_source_A_to_B_overlap_FINAL.json")


cross_source_A_sajid576_TO_B_sqliv2
{
  "external_before": 33537,
  "removed_normalized_overlap": 18355,
  "external_after": 15182,
  "external_remaining_pct": 45.2694039419149,
  "benign_after": 15004,
  "sqli_after": 178,
  "sqli_pct_after": 1.1724410486101964,
  "experiment": "cross_source_A_sajid576_TO_B_sqliv2",
  "train_source": "A_sajid576",
  "external_source": "B_sqliv2",
  "train_source_rows": 30764,
  "train_partition_rows": 27687,
  "val_partition_rows": 3077
}

cross_source_A_sajid576_TO_B_sqliv2 | CodeBERT | seed=42 | train=27,687, val=3,077, test=15,182


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4: loss=0.068443, val_f1=0.998672, val_acc=0.999025
Epoch 2/4: loss=0.003634, val_f1=1.000000, val_acc=1.000000
Epoch 3/4: loss=0.001430, val_f1=0.999558, val_acc=0.999675
Epoch 4/4: loss=0.000467, val_f1=0.999558, val_acc=0.999675
TEST: acc=0.949019, f1=0.308929, roc_auc=0.990252, pr_auc=0.839063, bal_acc=0.960329, mcc=0.410801, errors=774, FP=769, FN=5

=== FINAL A -> B RESULT ===
   model  seed  best_epoch  accuracy  precision  recall       f1  roc_auc   pr_auc  balanced_accuracy      mcc    tn  fp  fn  tp  errors  test_n  test_sqli  test_benign
CodeBERT    42           2  0.949019   0.183652 0.97191 0.308929 0.990252 0.839063           0.960329 0.410801 14235 769   5 173     774   15182        178        15004

Saved:
/kaggle/working/sqlshield_corrected_validation/cross_source_A_to_B_FINAL.csv
/kaggle/working/sqlshield_corrected_validation/cross_source_A_to_B_overlap_FINAL.json


In [18]:
import pandas as pd
import json
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

# ============================================================
# CROSS-SOURCE: B -> A
# ============================================================

ba_rows, ba_overlap = run_cross_direction(
    train_source=b_cross,
    external_source=a_cross,
    train_name="B_sqliv2",
    external_name="A_sajid576",
)

ba_df = pd.DataFrame(ba_rows)

print("\n=== FINAL B -> A RESULT ===")

cols = [
    "model",
    "seed",
    "best_epoch",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "mcc",
    "tn",
    "fp",
    "fn",
    "tp",
    "errors",
    "test_n",
    "test_sqli",
    "test_benign",
]

print(ba_df[cols].to_string(index=False))

ba_df.to_csv(
    SAVE_DIR / "cross_source_B_to_A_FINAL.csv",
    index=False
)

with open(
    SAVE_DIR / "cross_source_B_to_A_overlap_FINAL.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(ba_overlap, f, indent=2, default=str)

print("\nSaved:")
print(SAVE_DIR / "cross_source_B_to_A_FINAL.csv")
print(SAVE_DIR / "cross_source_B_to_A_overlap_FINAL.json")


cross_source_B_sqliv2_TO_A_sajid576
{
  "external_before": 30764,
  "removed_normalized_overlap": 18356,
  "external_after": 12408,
  "external_remaining_pct": 40.33285658561955,
  "benign_after": 12313,
  "sqli_after": 95,
  "sqli_pct_after": 0.7656350741457124,
  "experiment": "cross_source_B_sqliv2_TO_A_sajid576",
  "train_source": "B_sqliv2",
  "external_source": "A_sajid576",
  "train_source_rows": 33537,
  "train_partition_rows": 30183,
  "val_partition_rows": 3354
}

cross_source_B_sqliv2_TO_A_sajid576 | CodeBERT | seed=42 | train=30,183, val=3,354, test=12,408


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4: loss=0.069166, val_f1=0.997364, val_acc=0.998211
Epoch 2/4: loss=0.004628, val_f1=0.999120, val_acc=0.999404
Epoch 3/4: loss=0.000924, val_f1=0.999120, val_acc=0.999404
Epoch 4/4: loss=0.000422, val_f1=0.999120, val_acc=0.999404
TEST: acc=0.015474, f1=0.015315, roc_auc=0.703293, pr_auc=0.034568, bal_acc=0.503939, mcc=0.007797, errors=12216, FP=12216, FN=0

=== FINAL B -> A RESULT ===
   model  seed  best_epoch  accuracy  precision  recall       f1  roc_auc   pr_auc  balanced_accuracy      mcc  tn    fp  fn  tp  errors  test_n  test_sqli  test_benign
CodeBERT    42           2  0.015474   0.007717     1.0 0.015315 0.703293 0.034568           0.503939 0.007797  97 12216   0  95   12216   12408         95        12313

Saved:
/kaggle/working/sqlshield_corrected_validation/cross_source_B_to_A_FINAL.csv
/kaggle/working/sqlshield_corrected_validation/cross_source_B_to_A_overlap_FINAL.json


In [19]:
from pathlib import Path
import pandas as pd
import numpy as np

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")
PRED_DIR = SAVE_DIR / "predictions"

# Find the B -> A prediction file automatically
matches = []

for p in PRED_DIR.glob("*.csv"):
    try:
        d = pd.read_csv(p)

        if (
            "true_label" in d.columns
            and "prob_sqli" in d.columns
            and len(d) == 12408
            and int(d["true_label"].sum()) == 95
        ):
            matches.append(p)
    except:
        pass

print("Matching files:")
for p in matches:
    print(p)

assert len(matches) >= 1, "B->A prediction file not found."

pred_path = matches[-1]
pred = pd.read_csv(pred_path)

print("\nUsing:")
print(pred_path)

print("\n=== BASIC AUDIT ===")
print("N:", len(pred))
print("True labels:")
print(pred["true_label"].value_counts().sort_index())

print("\nPredicted labels:")
print(pred["pred_label"].value_counts().sort_index())

print(
    "\nPredicted SQLi rate:",
    pred["pred_label"].mean()
)

# ============================================================
# SCORE DISTRIBUTION
# ============================================================

print("\n=== PROBABILITY DISTRIBUTION BY TRUE CLASS ===")

for label, name in [(0, "BENIGN"), (1, "SQLi")]:
    x = pred.loc[
        pred["true_label"] == label,
        "prob_sqli"
    ]

    print(f"\n{name}: n={len(x)}")
    print(
        x.quantile(
            [0, .01, .05, .10, .25, .50, .75, .90, .95, .99, 1]
        )
    )

# ============================================================
# FALSE POSITIVE EXAMPLES
# ============================================================

fp = pred[
    (pred["true_label"] == 0) &
    (pred["pred_label"] == 1)
].copy()

print("\n=== FALSE POSITIVES ===")
print("Count:", len(fp))

print("\nLowest-score false positives:")
print(
    fp.sort_values("prob_sqli")
      [["text", "prob_sqli"]]
      .head(20)
      .to_string(index=False)
)

print("\nHighest-score false positives:")
print(
    fp.sort_values("prob_sqli", ascending=False)
      [["text", "prob_sqli"]]
      .head(20)
      .to_string(index=False)
)

Matching files:
/kaggle/working/sqlshield_corrected_validation/predictions/cross_source_B_sqliv2_TO_A_sajid576__CodeBERT__seed42.csv

Using:
/kaggle/working/sqlshield_corrected_validation/predictions/cross_source_B_sqliv2_TO_A_sajid576__CodeBERT__seed42.csv

=== BASIC AUDIT ===
N: 12408
True labels:
true_label
0    12313
1       95
Name: count, dtype: int64

Predicted labels:
pred_label
0       97
1    12311
Name: count, dtype: int64

Predicted SQLi rate: 0.9921824629271437

=== PROBABILITY DISTRIBUTION BY TRUE CLASS ===

BENIGN: n=12313
0.00    0.000018
0.01    0.994367
0.05    0.998811
0.10    0.999577
0.25    0.999936
0.50    0.999972
0.75    0.999977
0.90    0.999978
0.95    0.999979
0.99    0.999980
1.00    0.999982
Name: prob_sqli, dtype: float64

SQLi: n=95
0.00    0.983104
0.01    0.995797
0.05    0.999713
0.10    0.999973
0.25    0.999975
0.50    0.999976
0.75    0.999979
0.90    0.999980
0.95    0.999981
0.99    0.999981
1.00    0.999982
Name: prob_sqli, dtype: float64

=== F

In [20]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from sklearn.metrics import (
    precision_recall_curve,
    roc_curve
)

SAVE_DIR = Path(
    "/kaggle/working/sqlshield_corrected_validation"
)

PRED_DIR = SAVE_DIR / "predictions"
CKPT_DIR = SAVE_DIR / "checkpoints"

SEED = 42
MODEL_ID = "microsoft/codebert-base"

# ============================================================
# 1. RECREATE EXACT B TRAIN/VALIDATION SPLIT
# ============================================================

b_train_check, b_val_check = source_train_val(b_cross)

print("=== B SOURCE SPLIT ===")
print("Train:", len(b_train_check))
print("Validation:", len(b_val_check))
print(
    "Validation labels:",
    b_val_check["label"].value_counts().to_dict()
)

# ============================================================
# 2. LOAD THE SAVED B -> A CHECKPOINT
# ============================================================

ckpt = (
    CKPT_DIR /
    "cross_source_B_sqliv2_TO_A_sajid576__CodeBERT__seed42.pt"
)

print("\nCheckpoint:")
print(ckpt)
print("Exists:", ckpt.exists())

assert ckpt.exists(), "Checkpoint not found."

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={
        0: "benign",
        1: "sqli"
    },
    label2id={
        "benign": 0,
        "sqli": 1
    },
).to(DEVICE)

model.load_state_dict(
    torch.load(
        ckpt,
        map_location=DEVICE
    )
)

val_loader = make_loader(
    b_val_check,
    tokenizer,
    shuffle=False,
    seed=SEED
)

# ============================================================
# 3. GET VALIDATION-B PROBABILITIES
# ============================================================

vm, _, val_y, val_prob = evaluate_transformer(
    model,
    val_loader
)

print("\n=== ORIGINAL VALIDATION-B PERFORMANCE ===")
print("Accuracy:", vm.accuracy)
print("Precision:", vm.precision)
print("Recall:", vm.recall)
print("F1:", vm.f1)
print("ROC-AUC:", vm.roc_auc)
print("PR-AUC:", vm.pr_auc)
print("Balanced accuracy:", vm.balanced_accuracy)
print("MCC:", vm.mcc)

# ============================================================
# 4. VALIDATION SCORE DISTRIBUTIONS
# ============================================================

print("\n=== VALIDATION-B PROBABILITY DISTRIBUTION ===")

for label, name in [
    (0, "BENIGN"),
    (1, "SQLi")
]:
    x = val_prob[val_y == label]

    print(f"\n{name}: n={len(x)}")

    print(
        pd.Series(x).quantile(
            [
                0,
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99,
                1
            ]
        )
    )

# ============================================================
# 5. CHOOSE THRESHOLD USING VALIDATION B ONLY
# ============================================================

# ---- Threshold maximizing validation F1 ----

precision, recall, thresholds_pr = precision_recall_curve(
    val_y,
    val_prob
)

f1_values = (
    2 * precision[:-1] * recall[:-1]
    /
    np.maximum(
        precision[:-1] + recall[:-1],
        1e-12
    )
)

idx_f1 = int(np.nanargmax(f1_values))
threshold_f1 = float(thresholds_pr[idx_f1])

# ---- Threshold maximizing Youden J / balanced accuracy ----

fpr, tpr, thresholds_roc = roc_curve(
    val_y,
    val_prob
)

youden = tpr - fpr

finite = np.isfinite(thresholds_roc)

finite_indices = np.where(finite)[0]

best_local = np.argmax(
    youden[finite]
)

idx_bal = finite_indices[best_local]

threshold_bal = float(
    thresholds_roc[idx_bal]
)

print("\n=== THRESHOLDS CHOSEN ON VALIDATION B ONLY ===")
print("Default threshold:", 0.5)
print("Best validation F1 threshold:", threshold_f1)
print(
    "Best validation balanced-accuracy threshold:",
    threshold_bal
)

# ============================================================
# 6. LOAD EXTERNAL A PREDICTIONS
# ============================================================

external_path = (
    PRED_DIR /
    "cross_source_B_sqliv2_TO_A_sajid576__CodeBERT__seed42.csv"
)

external = pd.read_csv(external_path)

ext_y = external["true_label"].to_numpy()
ext_prob = external["prob_sqli"].to_numpy()

# ============================================================
# 7. APPLY THRESHOLDS TO A WITHOUT TUNING ON A
# ============================================================

results = []

threshold_cases = {
    "default_0.5": 0.5,
    "validation_B_best_F1": threshold_f1,
    "validation_B_best_balanced_acc": threshold_bal,
}

for name, threshold in threshold_cases.items():

    ext_pred = (
        ext_prob >= threshold
    ).astype(int)

    m = compute_metrics(
        ext_y,
        ext_pred,
        ext_prob
    )

    results.append(
        {
            "threshold_source": name,
            "threshold": threshold,
            "accuracy": m.accuracy,
            "precision": m.precision,
            "recall": m.recall,
            "f1": m.f1,
            "roc_auc": m.roc_auc,
            "pr_auc": m.pr_auc,
            "balanced_accuracy":
                m.balanced_accuracy,
            "mcc": m.mcc,
            "tn": m.tn,
            "fp": m.fp,
            "fn": m.fn,
            "tp": m.tp,
            "errors": m.errors,
        }
    )

threshold_results = pd.DataFrame(results)

print(
    "\n=== B -> A THRESHOLD TRANSFER RESULTS ==="
)

print(
    threshold_results.to_string(
        index=False
    )
)

threshold_results.to_csv(
    SAVE_DIR /
    "cross_source_B_to_A_threshold_transfer.csv",
    index=False
)

print("\nSaved:")
print(
    SAVE_DIR /
    "cross_source_B_to_A_threshold_transfer.csv"
)

# Cleanup
del model
del tokenizer

if torch.cuda.is_available():
    torch.cuda.empty_cache()

=== B SOURCE SPLIT ===
Train: 30183
Validation: 3354
Validation labels: {0: 2217, 1: 1137}

Checkpoint:
/kaggle/working/sqlshield_corrected_validation/checkpoints/cross_source_B_sqliv2_TO_A_sajid576__CodeBERT__seed42.pt
Exists: True


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== ORIGINAL VALIDATION-B PERFORMANCE ===
Accuracy: 0.9994036970781157
Precision: 1.0
Recall: 0.9982409850483729
F1: 0.9991197183098591
ROC-AUC: 0.9996068597616007
PR-AUC: 0.999588216325887
Balanced accuracy: 0.9991204925241864
MCC: 0.9986697469963128

=== VALIDATION-B PROBABILITY DISTRIBUTION ===

BENIGN: n=2217
0.00    0.000018
0.01    0.000019
0.05    0.000019
0.10    0.000019
0.25    0.000019
0.50    0.000020
0.75    0.000020
0.90    0.000022
0.95    0.000024
0.99    0.000037
1.00    0.004050
dtype: float64

SQLi: n=1137
0.00    0.000020
0.01    0.999975
0.05    0.999979
0.10    0.999979
0.25    0.999980
0.50    0.999980
0.75    0.999981
0.90    0.999981
0.95    0.999981
0.99    0.999982
1.00    0.999982
dtype: float64

=== THRESHOLDS CHOSEN ON VALIDATION B ONLY ===
Default threshold: 0.5
Best validation F1 threshold: 0.9215250015258789
Best validation balanced-accuracy threshold: 0.9215250015258789

=== B -> A THRESHOLD TRANSFER RESULTS ===
              threshold_source  thresho

In [21]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")

# ============================================================
# 1. GET OVERLAP-FREE UNIQUE PARTS OF BOTH SOURCES
# ============================================================

# A patterns that do not exist in B
a_unique, a_report = residual_external(
    b_cross,
    a_cross
)

# B patterns that do not exist in A
b_unique, b_report = residual_external(
    a_cross,
    b_cross
)

# Benign only — this is the class responsible for the B->A collapse
a_benign = (
    a_unique[a_unique["label"] == 0]
    .copy()
    .reset_index(drop=True)
)

b_benign = (
    b_unique[b_unique["label"] == 0]
    .copy()
    .reset_index(drop=True)
)

print("=== OVERLAP-FREE BENIGN SETS ===")
print("A benign:", len(a_benign))
print("B benign:", len(b_benign))


# ============================================================
# 2. SIMPLE STRUCTURAL FEATURES
# ============================================================

def structural_features(df, source_name):
    x = df[["text"]].copy()

    s = x["text"].astype(str)

    x["source"] = source_name

    x["char_length"] = s.str.len()

    x["word_count"] = s.str.split().str.len()

    x["digit_count"] = s.str.count(r"\d")

    x["quote_count"] = (
        s.str.count("'") +
        s.str.count('"')
    )

    x["paren_count"] = (
        s.str.count(r"\(") +
        s.str.count(r"\)")
    )

    x["semicolon"] = (
        s.str.contains(";", regex=False)
        .astype(int)
    )

    x["starts_select"] = (
        s.str.match(r"(?i)^\s*select\b")
        .astype(int)
    )

    x["starts_insert"] = (
        s.str.match(r"(?i)^\s*insert\b")
        .astype(int)
    )

    x["starts_update"] = (
        s.str.match(r"(?i)^\s*update\b")
        .astype(int)
    )

    x["starts_delete"] = (
        s.str.match(r"(?i)^\s*delete\b")
        .astype(int)
    )

    patterns = {
        "has_where": r"\bwhere\b",
        "has_from": r"\bfrom\b",
        "has_union": r"\bunion\b",
        "has_join": r"\bjoin\b",
        "has_or": r"\bor\b",
        "has_and": r"\band\b",
        "has_group_by": r"\bgroup\s+by\b",
        "has_order_by": r"\border\s+by\b",
        "has_comment_dash": r"--",
        "has_comment_hash": r"#",
        "has_comment_block": r"/\*",
    }

    for col, pattern in patterns.items():
        x[col] = (
            s.str.contains(
                pattern,
                case=False,
                regex=True
            )
            .astype(int)
        )

    return x


fa = structural_features(
    a_benign,
    "A"
)

fb = structural_features(
    b_benign,
    "B"
)

features = pd.concat(
    [fa, fb],
    ignore_index=True
)


# ============================================================
# 3. COMPARE STRUCTURAL DISTRIBUTIONS
# ============================================================

continuous = [
    "char_length",
    "word_count",
    "digit_count",
    "quote_count",
    "paren_count",
]

binary = [
    "semicolon",
    "starts_select",
    "starts_insert",
    "starts_update",
    "starts_delete",
    "has_where",
    "has_from",
    "has_union",
    "has_join",
    "has_or",
    "has_and",
    "has_group_by",
    "has_order_by",
    "has_comment_dash",
    "has_comment_hash",
    "has_comment_block",
]

rows = []

for col in continuous:
    for src in ["A", "B"]:
        v = features.loc[
            features["source"] == src,
            col
        ]

        rows.append({
            "feature": col,
            "source": src,
            "type": "continuous",
            "mean": v.mean(),
            "median": v.median(),
            "std": v.std(),
            "rate_pct": np.nan,
        })

for col in binary:
    for src in ["A", "B"]:
        v = features.loc[
            features["source"] == src,
            col
        ]

        rows.append({
            "feature": col,
            "source": src,
            "type": "binary",
            "mean": v.mean(),
            "median": np.nan,
            "std": np.nan,
            "rate_pct": 100 * v.mean(),
        })

structural_summary = pd.DataFrame(rows)

print("\n=== STRUCTURAL SUMMARY ===")

pivot_rates = (
    structural_summary[
        structural_summary["type"] == "binary"
    ]
    .pivot(
        index="feature",
        columns="source",
        values="rate_pct"
    )
)

pivot_rates["abs_difference_pct"] = (
    pivot_rates["A"] -
    pivot_rates["B"]
).abs()

print("\n--- Largest binary-feature differences ---")

print(
    pivot_rates
    .sort_values(
        "abs_difference_pct",
        ascending=False
    )
    .head(15)
    .to_string()
)

print("\n--- Length statistics ---")

for col in continuous:
    print(f"\n{col}")

    print(
        features.groupby("source")[col]
        .agg(
            [
                "count",
                "mean",
                "median",
                "std",
                "min",
                "max",
            ]
        )
        .to_string()
    )


# ============================================================
# 4. CAN TEXT ALONE IDENTIFY THE SOURCE?
# ============================================================

source_df = pd.concat(
    [
        a_benign[
            ["text", "normalized_text"]
        ].assign(source_label=0),

        b_benign[
            ["text", "normalized_text"]
        ].assign(source_label=1),
    ],
    ignore_index=True
)

# Keep one row per normalized pattern
source_df = (
    source_df
    .drop_duplicates(
        subset=["normalized_text"]
    )
    .reset_index(drop=True)
)

train_src, test_src = train_test_split(
    source_df,
    test_size=0.20,
    stratify=source_df["source_label"],
    random_state=42,
)

# Character n-grams are deliberately used here
# because they reveal formatting/source artifacts.
vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=30000,
    min_df=2,
)

X_train = vectorizer.fit_transform(
    train_src["text"]
)

X_test = vectorizer.transform(
    test_src["text"]
)

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
)

clf.fit(
    X_train,
    train_src["source_label"]
)

pred = clf.predict(X_test)

prob = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(
    test_src["source_label"],
    pred
)

bal = balanced_accuracy_score(
    test_src["source_label"],
    pred
)

f1 = f1_score(
    test_src["source_label"],
    pred
)

auc = roc_auc_score(
    test_src["source_label"],
    prob
)

tn, fp, fn, tp = confusion_matrix(
    test_src["source_label"],
    pred,
    labels=[0, 1]
).ravel()

print("\n=== BENIGN SOURCE-IDENTIFICATION TEST ===")

print("Task: identify whether benign query came from A or B")
print("Accuracy:", acc)
print("Balanced accuracy:", bal)
print("F1:", f1)
print("ROC-AUC:", auc)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)


# ============================================================
# 5. MOST SOURCE-SPECIFIC CHARACTER PATTERNS
# ============================================================

feature_names = np.array(
    vectorizer.get_feature_names_out()
)

coef = clf.coef_[0]

top_b_idx = np.argsort(coef)[-30:][::-1]
top_a_idx = np.argsort(coef)[:30]

top_b = pd.DataFrame({
    "ngram": feature_names[top_b_idx],
    "coefficient": coef[top_b_idx],
    "associated_source": "B",
})

top_a = pd.DataFrame({
    "ngram": feature_names[top_a_idx],
    "coefficient": coef[top_a_idx],
    "associated_source": "A",
})

source_ngrams = pd.concat(
    [top_a, top_b],
    ignore_index=True
)

print("\n=== TOP SOURCE-A CHARACTER PATTERNS ===")
print(
    top_a.head(20)
    .to_string(index=False)
)

print("\n=== TOP SOURCE-B CHARACTER PATTERNS ===")
print(
    top_b.head(20)
    .to_string(index=False)
)


# ============================================================
# 6. SAVE EVERYTHING
# ============================================================

structural_summary.to_csv(
    SAVE_DIR /
    "cross_source_structural_diagnostic.csv",
    index=False
)

source_ngrams.to_csv(
    SAVE_DIR /
    "cross_source_source_specific_ngrams.csv",
    index=False
)

source_classifier_result = pd.DataFrame([{
    "task":
        "benign_A_vs_B_source_identification",
    "train_n":
        len(train_src),
    "test_n":
        len(test_src),
    "accuracy":
        acc,
    "balanced_accuracy":
        bal,
    "f1":
        f1,
    "roc_auc":
        auc,
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tp": tp,
}])

source_classifier_result.to_csv(
    SAVE_DIR /
    "cross_source_source_classifier.csv",
    index=False
)

print("\n=== SAVED ===")
print(
    SAVE_DIR /
    "cross_source_structural_diagnostic.csv"
)
print(
    SAVE_DIR /
    "cross_source_source_classifier.csv"
)
print(
    SAVE_DIR /
    "cross_source_source_specific_ngrams.csv"
)

=== OVERLAP-FREE BENIGN SETS ===
A benign: 12313
B benign: 15004

=== STRUCTURAL SUMMARY ===

--- Largest binary-feature differences ---
source                     A         B  abs_difference_pct
feature                                                   
starts_select      87.614716  0.000000           87.614716
has_from           83.594575  0.046654           83.547921
has_where          35.612767  0.013330           35.599437
has_and            10.338666  0.279925           10.058741
has_order_by        8.803703  0.000000            8.803703
has_join            7.463656  0.026660            7.436997
has_union           4.572403  0.039989            4.532414
has_or              2.948104  0.039989            2.908114
starts_update       1.786729  0.000000            1.786729
starts_insert       1.778608  0.000000            1.778608
starts_delete       1.770486  0.000000            1.770486
semicolon           4.710469  6.358304            1.647836
has_group_by        0.138065  0.00000

In [22]:
from pathlib import Path
import pandas as pd
import numpy as np

SAVE_DIR = Path("/kaggle/working/sqlshield_corrected_validation")
PRED_DIR = SAVE_DIR / "predictions"

files = {
    "CodeBERT": PRED_DIR / "corrected_group_split_pilot__CodeBERT__seed42.csv",
    "BERT-base": PRED_DIR / "corrected_group_split_pilot__BERT-base__seed42.csv",
    "Random Forest": PRED_DIR / "classical__Random_Forest.csv",
}

rows = []

print("=== FILE CHECK ===")
for model, path in files.items():
    print(model, "->", path.exists(), path)

for model, path in files.items():
    assert path.exists(), f"Missing: {path}"

    d = pd.read_csv(path)

    score_col = (
        "prob_sqli"
        if "prob_sqli" in d.columns
        else "score_sqli"
    )

    print(f"\n=== {model}: SOURCE / LABEL DISTRIBUTION ===")
    print(
        pd.crosstab(
            d["source"],
            d["true_label"],
            margins=True
        )
    )

    for source, g in d.groupby("source"):

        m = compute_metrics(
            g["true_label"],
            g["pred_label"],
            g[score_col]
        )

        rows.append({
            "model": model,
            "source": source,
            "n": len(g),
            "benign": int((g["true_label"] == 0).sum()),
            "sqli": int((g["true_label"] == 1).sum()),
            "accuracy": m.accuracy,
            "precision": m.precision,
            "recall": m.recall,
            "f1": m.f1,
            "roc_auc": m.roc_auc,
            "pr_auc": m.pr_auc,
            "balanced_accuracy": m.balanced_accuracy,
            "mcc": m.mcc,
            "tn": m.tn,
            "fp": m.fp,
            "fn": m.fn,
            "tp": m.tp,
            "errors": m.errors,
        })

audit = pd.DataFrame(rows)

print("\n=== SOURCE-CONDITIONED FIXED-TEST PERFORMANCE ===")
print(
    audit[
        [
            "model",
            "source",
            "n",
            "benign",
            "sqli",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "balanced_accuracy",
            "mcc",
            "errors",
            "fp",
            "fn",
        ]
    ].to_string(index=False)
)

audit.to_csv(
    SAVE_DIR / "source_conditioned_test_performance_FINAL.csv",
    index=False
)

print("\nSaved:")
print(SAVE_DIR / "source_conditioned_test_performance_FINAL.csv")

=== FILE CHECK ===
CodeBERT -> True /kaggle/working/sqlshield_corrected_validation/predictions/corrected_group_split_pilot__CodeBERT__seed42.csv
BERT-base -> True /kaggle/working/sqlshield_corrected_validation/predictions/corrected_group_split_pilot__BERT-base__seed42.csv
Random Forest -> True /kaggle/working/sqlshield_corrected_validation/predictions/classical__Random_Forest.csv

=== CodeBERT: SOURCE / LABEL DISTRIBUTION ===
true_label                                             0     1   All
source                                                              
sajid576/sql-injection-dataset                      1990  1128  3118
syedsaqlainhussain/sql-injection-dataset/sqliv2...  1460  1076  2536
All                                                 3450  2204  5654

=== BERT-base: SOURCE / LABEL DISTRIBUTION ===
true_label                                             0     1   All
source                                                              
sajid576/sql-injection-dataset         

In [2]:
!kaggle kernels output mohammadalkhazaleh/notebook691d0eee69 \
-p /kaggle/working/version8_restore

Output file downloaded to /kaggle/working/version8_restore/.virtual_documents/__notebook_source__.ipynb
Output file downloaded to /kaggle/working/version8_restore/sqlshield_corrected_validation/bert_seed126_result.json
Output file downloaded to /kaggle/working/version8_restore/sqlshield_corrected_validation/bert_seed42_result.json
Output file downloaded to /kaggle/working/version8_restore/sqlshield_corrected_validation/checkpoints/corrected_group_split_multiseed__BERT-base__seed126.pt
Output file downloaded to /kaggle/working/version8_restore/sqlshield_corrected_validation/checkpoints/corrected_group_split_multiseed__CodeBERT__seed21.pt
Output file downloaded to /kaggle/working/version8_restore/sqlshield_corrected_validation/checkpoints/corrected_group_split_pilot__BERT-base__seed42.pt
Output file downloaded to /kaggle/working/version8_restore/sqlshield_corrected_validation/checkpoints/corrected_group_split_pilot__CodeBERT__seed42.pt
Output file downloaded to /kaggle/working/version8_r

In [3]:
from pathlib import Path

root = Path("/kaggle/working/version8_restore")

print("=== VERSION 8 FILES ===")
for f in sorted(root.rglob("*")):
    if f.is_file():
        print(f)

=== VERSION 8 FILES ===
/kaggle/working/version8_restore/.virtual_documents/__notebook_source__.ipynb
/kaggle/working/version8_restore/notebook691d0eee69.log
/kaggle/working/version8_restore/sqlshield_corrected_validation/bert_seed126_result.json
/kaggle/working/version8_restore/sqlshield_corrected_validation/bert_seed42_result.json
/kaggle/working/version8_restore/sqlshield_corrected_validation/checkpoints/corrected_group_split_multiseed__BERT-base__seed126.pt
/kaggle/working/version8_restore/sqlshield_corrected_validation/checkpoints/corrected_group_split_multiseed__CodeBERT__seed21.pt
/kaggle/working/version8_restore/sqlshield_corrected_validation/checkpoints/corrected_group_split_pilot__BERT-base__seed42.pt
/kaggle/working/version8_restore/sqlshield_corrected_validation/checkpoints/corrected_group_split_pilot__CodeBERT__seed42.pt
/kaggle/working/version8_restore/sqlshield_corrected_validation/checkpoints/cross_source_A_sajid576_TO_B_sqliv2__CodeBERT__seed42.pt
/kaggle/working/versi

In [4]:
import shutil
from pathlib import Path

SRC = Path(
    "/kaggle/working/version8_restore/sqlshield_corrected_validation"
)

DST = Path(
    "/kaggle/working/sqlshield_corrected_validation"
)

assert SRC.exists(), "Version 8 restore folder not found!"

if DST.exists():
    shutil.rmtree(DST)

shutil.copytree(SRC, DST)

print("Restored Version 8 to:")
print(DST)

print("\n=== KEY FILE CHECK ===")

required = [
    "train_corrected.csv",
    "val_corrected.csv",
    "test_corrected.csv",
    "codebert_seed42_result.json",
    "bert_seed42_result.json",
    "transformer_multiseed_runs_FINAL.csv",
    "transformer_multiseed_summary_FINAL.csv",
    "cross_source_A_to_B_FINAL.csv",
    "cross_source_B_to_A_FINAL.csv",
    "cross_source_B_to_A_threshold_transfer.csv",
    "cross_source_source_classifier.csv",
    "cross_source_structural_diagnostic.csv",
    "source_conditioned_test_performance_FINAL.csv",
]

for name in required:
    p = DST / name
    print(name, "->", p.exists())

assert all((DST / x).exists() for x in required)

print("\nVERSION 8 RESTORE: VERIFIED")

Restored Version 8 to:
/kaggle/working/sqlshield_corrected_validation

=== KEY FILE CHECK ===
train_corrected.csv -> True
val_corrected.csv -> True
test_corrected.csv -> True
codebert_seed42_result.json -> True
bert_seed42_result.json -> True
transformer_multiseed_runs_FINAL.csv -> True
transformer_multiseed_summary_FINAL.csv -> True
cross_source_A_to_B_FINAL.csv -> True
cross_source_B_to_A_FINAL.csv -> True
cross_source_B_to_A_threshold_transfer.csv -> True
cross_source_source_classifier.csv -> True
cross_source_structural_diagnostic.csv -> True
source_conditioned_test_performance_FINAL.csv -> True

VERSION 8 RESTORE: VERIFIED


In [5]:
from pathlib import Path
import pandas as pd

SAVE_DIR = Path(
    "/kaggle/working/sqlshield_corrected_validation"
)

# Load EXACT saved split from Version 8
train_df = pd.read_csv(
    SAVE_DIR / "train_corrected.csv"
)

test_df = pd.read_csv(
    SAVE_DIR / "test_corrected.csv"
)

print("Train:", len(train_df))
print("Test:", len(test_df))

assert len(train_df) == 45288
assert len(test_df) == 5654

print("\nRunning all four classical baselines...")

classical_results_FINAL = run_classical(
    train_df,
    test_df
)

# Save explicitly as FINAL
classical_results_FINAL.to_csv(
    SAVE_DIR / "classical_results_FINAL.csv",
    index=False
)

print("\n=== FINAL CLASSICAL RESULTS ===")

cols = [
    "model",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "mcc",
    "tn",
    "fp",
    "fn",
    "tp",
    "errors",
]

print(
    classical_results_FINAL[cols]
    .sort_values("f1", ascending=False)
    .to_string(index=False)
)

print("\nSaved:")
print(SAVE_DIR / "classical_results_FINAL.csv")

Train: 45288
Test: 5654

Running all four classical baselines...

CLASSICAL BASELINES ON CORRECTED GROUP-AWARE SPLIT

Training Random Forest ...
Random Forest: acc=0.997170, f1=0.996365, roc_auc=0.998884, errors=16, FP=5, FN=11

Training XGBoost ...
XGBoost: acc=0.996286, f1=0.995224, roc_auc=0.998267, errors=21, FP=5, FN=16

Training Logistic Regression ...
Logistic Regression: acc=0.990803, f1=0.988144, roc_auc=0.998396, errors=52, FP=15, FN=37

Training SVM ...
SVM: acc=0.995578, f1=0.994322, roc_auc=0.998418, errors=25, FP=10, FN=15

=== FINAL CLASSICAL RESULTS ===
              model  accuracy  precision   recall       f1  roc_auc   pr_auc  balanced_accuracy      mcc   tn  fp  fn   tp  errors
      Random Forest  0.997170   0.997725 0.995009 0.996365 0.998884 0.998386           0.996780 0.994051 3445   5  11 2193      16
            XGBoost  0.996286   0.997720 0.992740 0.995224 0.998267 0.997197           0.995646 0.992194 3445   5  16 2188      21
                SVM  0.995578  

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import binomtest

SAVE_DIR = Path(
    "/kaggle/working/sqlshield_corrected_validation"
)
PRED_DIR = SAVE_DIR / "predictions"

# ============================================================
# LOAD FIXED-TEST PREDICTIONS
# ============================================================

paths = {
    "CodeBERT": (
        PRED_DIR /
        "corrected_group_split_pilot__CodeBERT__seed42.csv"
    ),
    "BERT-base": (
        PRED_DIR /
        "corrected_group_split_pilot__BERT-base__seed42.csv"
    ),
    "Random Forest": (
        PRED_DIR /
        "classical__Random_Forest.csv"
    ),
    "XGBoost": (
        PRED_DIR /
        "classical__XGBoost.csv"
    ),
    "SVM": (
        PRED_DIR /
        "classical__SVM.csv"
    ),
    "Logistic Regression": (
        PRED_DIR /
        "classical__Logistic_Regression.csv"
    ),
}

print("=== FILE CHECK ===")

for name, path in paths.items():
    print(name, "->", path.exists(), path)
    assert path.exists(), f"Missing: {path}"

dfs = {
    name: pd.read_csv(path)
    for name, path in paths.items()
}

# ============================================================
# STRICT SAME-TEST VERIFICATION
# ============================================================

reference = dfs["CodeBERT"]

assert len(reference) == 5654

for name, d in dfs.items():

    assert len(d) == 5654, f"N mismatch: {name}"

    assert (
        d["text"].astype(str).values ==
        reference["text"].astype(str).values
    ).all(), f"Text mismatch: {name}"

    assert (
        d["true_label"].values ==
        reference["true_label"].values
    ).all(), f"Label mismatch: {name}"

print("\nSame fixed test set for all models: VERIFIED")

# ============================================================
# EXACT McNEMAR
# ============================================================

def exact_mcnemar(df1, df2, name1, name2):

    y = df1["true_label"].values

    correct1 = (
        df1["pred_label"].values == y
    )

    correct2 = (
        df2["pred_label"].values == y
    )

    # model1 wrong, model2 correct
    b = int(
        ((~correct1) & correct2).sum()
    )

    # model1 correct, model2 wrong
    c = int(
        (correct1 & (~correct2)).sum()
    )

    discordant = b + c

    if discordant == 0:
        p = 1.0
    else:
        p = binomtest(
            min(b, c),
            n=discordant,
            p=0.5,
            alternative="two-sided"
        ).pvalue

    return {
        "comparison": f"{name1} vs {name2}",
        "model1_errors":
            int((~correct1).sum()),
        "model2_errors":
            int((~correct2).sum()),
        "model1_wrong_model2_correct": b,
        "model1_correct_model2_wrong": c,
        "discordant_pairs": discordant,
        "exact_p_value": float(p),
    }


comparisons = [
    ("CodeBERT", "BERT-base"),
    ("CodeBERT", "Random Forest"),
    ("CodeBERT", "XGBoost"),
    ("CodeBERT", "SVM"),
    ("CodeBERT", "Logistic Regression"),
]

rows = []

for m1, m2 in comparisons:
    rows.append(
        exact_mcnemar(
            dfs[m1],
            dfs[m2],
            m1,
            m2
        )
    )

mcnemar = pd.DataFrame(rows)

# ============================================================
# BONFERRONI CORRECTION
# 5 primary pairwise comparisons
# ============================================================

n_tests = len(mcnemar)
alpha = 0.05
bonf_alpha = alpha / n_tests

mcnemar["bonferroni_adjusted_p"] = np.minimum(
    mcnemar["exact_p_value"] * n_tests,
    1.0
)

mcnemar["significant_raw_0.05"] = (
    mcnemar["exact_p_value"] < 0.05
)

mcnemar["significant_bonferroni"] = (
    mcnemar["exact_p_value"] < bonf_alpha
)

# ============================================================
# HOLM CORRECTION
# ============================================================

pvals = mcnemar["exact_p_value"].to_numpy()
order = np.argsort(pvals)

holm_adj = np.empty(len(pvals), dtype=float)

running_max = 0.0

for rank, idx in enumerate(order):

    adjusted = (
        (len(pvals) - rank) *
        pvals[idx]
    )

    running_max = max(
        running_max,
        adjusted
    )

    holm_adj[idx] = min(
        running_max,
        1.0
    )

mcnemar["holm_adjusted_p"] = holm_adj

mcnemar["significant_holm"] = (
    mcnemar["holm_adjusted_p"] < 0.05
)

# ============================================================
# DISPLAY + SAVE
# ============================================================

print("\n=== FINAL EXACT McNEMAR RESULTS ===")
print(
    mcnemar.to_string(index=False)
)

print(
    "\nBonferroni family-wise alpha:",
    bonf_alpha
)

mcnemar.to_csv(
    SAVE_DIR / "exact_mcnemar_FINAL.csv",
    index=False
)

print("\nSaved:")
print(
    SAVE_DIR / "exact_mcnemar_FINAL.csv"
)

=== FILE CHECK ===
CodeBERT -> True /kaggle/working/sqlshield_corrected_validation/predictions/corrected_group_split_pilot__CodeBERT__seed42.csv
BERT-base -> True /kaggle/working/sqlshield_corrected_validation/predictions/corrected_group_split_pilot__BERT-base__seed42.csv
Random Forest -> True /kaggle/working/sqlshield_corrected_validation/predictions/classical__Random_Forest.csv
XGBoost -> True /kaggle/working/sqlshield_corrected_validation/predictions/classical__XGBoost.csv
SVM -> True /kaggle/working/sqlshield_corrected_validation/predictions/classical__SVM.csv
Logistic Regression -> True /kaggle/working/sqlshield_corrected_validation/predictions/classical__Logistic_Regression.csv

Same fixed test set for all models: VERIFIED

=== FINAL EXACT McNEMAR RESULTS ===
                     comparison  model1_errors  model2_errors  model1_wrong_model2_correct  model1_correct_model2_wrong  discordant_pairs  exact_p_value  bonferroni_adjusted_p  significant_raw_0.05  significant_bonferroni  h

In [1]:
from pathlib import Path

print("=== Looking for SQLShield results ===")

for p in Path("/kaggle/input").rglob("train_corrected.csv"):
    print("TRAIN:", p)

for p in Path("/kaggle/input").rglob("test_corrected.csv"):
    print("TEST:", p)

for p in Path("/kaggle/input").rglob("*CodeBERT*seed42*.csv"):
    print("CODEBERT:", p)

for p in Path("/kaggle/input").rglob("*BERT-base*seed42*.csv"):
    print("BERT:", p)

=== Looking for SQLShield results ===
TRAIN: /kaggle/input/notebooks/mohammadalkhazaleh/notebook691d0eee69/sqlshield_corrected_validation/train_corrected.csv
TRAIN: /kaggle/input/notebooks/mohammadalkhazaleh/notebook691d0eee69/version8_restore/sqlshield_corrected_validation/train_corrected.csv
TRAIN: /kaggle/input/notebooks/mohammadalkhazaleh/notebook691d0eee69/version8_restore/version4_restore/sqlshield_corrected_validation/train_corrected.csv
TEST: /kaggle/input/notebooks/mohammadalkhazaleh/notebook691d0eee69/sqlshield_corrected_validation/test_corrected.csv
TEST: /kaggle/input/notebooks/mohammadalkhazaleh/notebook691d0eee69/version8_restore/sqlshield_corrected_validation/test_corrected.csv
TEST: /kaggle/input/notebooks/mohammadalkhazaleh/notebook691d0eee69/version8_restore/version4_restore/sqlshield_corrected_validation/test_corrected.csv
CODEBERT: /kaggle/input/notebooks/mohammadalkhazaleh/notebook691d0eee69/sqlshield_corrected_validation/histories/corrected_group_split_pilot__Code

In [2]:
from pathlib import Path
import shutil

SRC = Path(
    "/kaggle/input/notebooks/mohammadalkhazaleh/"
    "notebook691d0eee69/sqlshield_corrected_validation"
)

DST = Path("/kaggle/working/sqlshield_corrected_validation")

# إنشاء مجلد عمل نظيف
if DST.exists():
    shutil.rmtree(DST)

DST.mkdir(parents=True, exist_ok=True)

# الملفات التي نحتاجها
for name in [
    "train_corrected.csv",
    "val_corrected.csv",
    "test_corrected.csv",
]:
    src_file = SRC / name
    if src_file.exists():
        shutil.copy2(src_file, DST / name)
        print("COPIED:", name)
    else:
        print("MISSING:", name)

# نسخ جميع predictions القديمة
shutil.copytree(
    SRC / "predictions",
    DST / "predictions"
)

print("\nDone.")
print("Working directory:", DST)

COPIED: train_corrected.csv
COPIED: val_corrected.csv
COPIED: test_corrected.csv

Done.
Working directory: /kaggle/working/sqlshield_corrected_validation


In [3]:
import pandas as pd
from pathlib import Path

base = Path("/kaggle/working/sqlshield_corrected_validation")

train = pd.read_csv(base / "train_corrected.csv")
test  = pd.read_csv(base / "test_corrected.csv")

print("Train:", len(train))
print("Test :", len(test))

print("\nImportant prediction files:")

for name in [
    "corrected_group_split_pilot__CodeBERT__seed42.csv",
    "corrected_group_split_pilot__BERT-base__seed42.csv",
    "classical__Random_Forest.csv",
]:
    p = base / "predictions" / name
    print(name, "->", p.exists())

Train: 45288
Test : 5654

Important prediction files:
corrected_group_split_pilot__CodeBERT__seed42.csv -> True
corrected_group_split_pilot__BERT-base__seed42.csv -> True
classical__Random_Forest.csv -> True


In [4]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("exp_char_ngram_baselines.py"):
    print(p)

/kaggle/input/datasets/mohammadalkhazaleh/sqlshield-char-ngram-script/exp_char_ngram_baselines.py


In [6]:
!cp /kaggle/input/datasets/mohammadalkhazaleh/sqlshield-char-ngram-script/exp_char_ngram_baselines.py \
    /kaggle/working/exp_char_ngram_baselines.py

In [7]:
from pathlib import Path

p = Path("/kaggle/working/exp_char_ngram_baselines.py")
print(p.exists(), p)

True /kaggle/working/exp_char_ngram_baselines.py


In [8]:
!python /kaggle/working/exp_char_ngram_baselines.py \
    --base-dir /kaggle/working/sqlshield_corrected_validation

SQLShield CHARACTER N-GRAM BASELINE EXTENSION
Base dir: /kaggle/working/sqlshield_corrected_validation
Train n: 45,288
Test  n: 5,654
Character TF-IDF: analyzer='char', ngram_range=(3,5), max_features=50000

Training Char LinearSVC ...
Char LinearSVC: F1=0.997956 | Acc=0.998408 | ROC-AUC=0.999587 | PR-AUC=0.999524 | BA=0.998122 | MCC=0.996654 | errors=9 (FP=2, FN=7) | 5.9s
Saved predictions: /kaggle/working/sqlshield_corrected_validation/predictions/classical__Char_LinearSVC.csv

Training Char Logistic Regression ...
Char Logistic Regression: F1=0.995448 | Acc=0.996463 | ROC-AUC=0.999347 | PR-AUC=0.999340 | BA=0.995709 | MCC=0.992569 | errors=20 (FP=3, FN=17) | 7.5s
Saved predictions: /kaggle/working/sqlshield_corrected_validation/predictions/classical__Char_Logistic_Regression.csv
Using existing prediction: classical__Random_Forest.csv
Using existing prediction: classical__XGBoost.csv
Using existing prediction: classical__SVM.csv
Using existing prediction: classical__Logistic_Regressi

In [9]:
from pathlib import Path
import shutil

base = Path("/kaggle/working/sqlshield_corrected_validation")

files_to_save = [
    "char_ngram_results.csv",
    "mcnemar_7comparisons_char_augmented.csv",
    "char_ngram_experiment_summary.json",
]

package = Path("/kaggle/working/sqlshield_char_extension")

if package.exists():
    shutil.rmtree(package)

package.mkdir()

for name in files_to_save:
    shutil.copy2(base / name, package / name)

# predictions الجديدة
pred_dir = package / "predictions"
pred_dir.mkdir()

for name in [
    "classical__Char_LinearSVC.csv",
    "classical__Char_Logistic_Regression.csv",
]:
    shutil.copy2(base / "predictions" / name, pred_dir / name)

# ZIP
shutil.make_archive(
    "/kaggle/working/sqlshield_char_extension_results",
    "zip",
    package
)

print("/kaggle/working/sqlshield_char_extension_results.zip")

/kaggle/working/sqlshield_char_extension_results.zip


In [10]:
from IPython.display import FileLink

FileLink('/kaggle/working/sqlshield_char_extension_results.zip')

/kaggle/working/sqlshield_char_extension_results.zip

In [11]:
from pathlib import Path

p = Path("/kaggle/working/sqlshield_char_extension_results.zip")

print("Exists:", p.exists())
if p.exists():
    print("Size:", p.stat().st_size, "bytes")
    print("Path:", p)

Exists: True
Size: 316224 bytes
Path: /kaggle/working/sqlshield_char_extension_results.zip


In [12]:
!pip install -q datasketch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 3.2 MB/s eta 0:00:00


In [13]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("exp_fuzzy_group_sensitivity.py"):
    print(p)

/kaggle/input/datasets/mohammadalkhazaleh/sqlshield-fuzzy-sensitivity-script/exp_fuzzy_group_sensitivity.py


In [15]:
!cp /kaggle/input/datasets/mohammadalkhazaleh/sqlshield-fuzzy-sensitivity-script/exp_fuzzy_group_sensitivity.py \
    /kaggle/working/exp_fuzzy_group_sensitivity.py

In [16]:
from pathlib import Path

p = Path("/kaggle/working/exp_fuzzy_group_sensitivity.py")
print(p.exists(), p)

True /kaggle/working/exp_fuzzy_group_sensitivity.py


In [17]:
!python /kaggle/working/exp_fuzzy_group_sensitivity.py \
    --base-dir /kaggle/working/sqlshield_corrected_validation \
    --cluster-only


SQLShield FUZZY / NEAR-DUPLICATE SENSITIVITY EXPERIMENT
Base dir:       /kaggle/working/sqlshield_corrected_validation
Output dir:     /kaggle/working/sqlshield_corrected_validation/fuzzy_sensitivity
Thresholds:     baseline 1.0 + [0.9, 0.8, 0.7]
Shingles:       5-char
MinHash perms:  256
LSH margin:     0.05
CodeBERT seed:  42
Cluster only:   True

Reconstructed corrected corpus: 56,621 rows
Exact normalized groups:        45,915
Class counts: benign=34,490 | SQLi=22,131

Preparing 45,915 exact normalized groups | 5-char shingles | num_perm=256
  MinHash prepared: 5,000/45,915 (0.0 min)
  MinHash prepared: 10,000/45,915 (0.1 min)
  MinHash prepared: 15,000/45,915 (0.1 min)
  MinHash prepared: 20,000/45,915 (0.1 min)
  MinHash prepared: 25,000/45,915 (0.2 min)
  MinHash prepared: 30,000/45,915 (0.2 min)
  MinHash prepared: 35,000/45,915 (0.2 min)
  MinHash prepared: 40,000/45,915 (0.2 min)
  MinHash prepared: 45,000/45,915 (0.3 min)
  MinHash prepared: 45,915/45,915 (0.3 min)

FUZZY C

In [18]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("exp_fuzzy_group_sensitivity_v2.py"):
    print(p)

/kaggle/input/datasets/mohammadalkhazaleh/sqlshield-fuzzy-sensitivity-v2/exp_fuzzy_group_sensitivity_v2.py


In [20]:
!cp /kaggle/input/datasets/mohammadalkhazaleh/sqlshield-fuzzy-sensitivity-v2/exp_fuzzy_group_sensitivity_v2.py \
    /kaggle/working/exp_fuzzy_group_sensitivity_v2.py

In [21]:
from pathlib import Path

p = Path("/kaggle/working/exp_fuzzy_group_sensitivity_v2.py")
print(p.exists(), p)

True /kaggle/working/exp_fuzzy_group_sensitivity_v2.py


In [22]:
!python /kaggle/working/exp_fuzzy_group_sensitivity_v2.py \
    --base-dir /kaggle/working/sqlshield_corrected_validation \
    --cluster-only


SQLShield FUZZY / NEAR-DUPLICATE SENSITIVITY EXPERIMENT
Base dir:       /kaggle/working/sqlshield_corrected_validation
Output dir:     /kaggle/working/sqlshield_corrected_validation/fuzzy_sensitivity
Thresholds:     baseline 1.0 + [0.9, 0.8, 0.7]
Shingles:       5-char
MinHash perms:  256
LSH margin:     0.05
CodeBERT seed:  42
Cluster only:   True

Reconstructed corrected corpus: 56,621 rows
Exact normalized groups:        45,915
Class counts: benign=34,490 | SQLi=22,131

Preparing 45,915 exact normalized groups | 5-char shingles | num_perm=256
  MinHash prepared: 5,000/45,915 (0.0 min)
  MinHash prepared: 10,000/45,915 (0.1 min)
  MinHash prepared: 15,000/45,915 (0.1 min)
  MinHash prepared: 20,000/45,915 (0.1 min)
  MinHash prepared: 25,000/45,915 (0.2 min)
  MinHash prepared: 30,000/45,915 (0.2 min)
  MinHash prepared: 35,000/45,915 (0.2 min)
  MinHash prepared: 40,000/45,915 (0.2 min)
  MinHash prepared: 45,000/45,915 (0.3 min)
  MinHash prepared: 45,915/45,915 (0.3 min)

FUZZY C

In [23]:
!cp /kaggle/input/datasets/mohammadalkhazaleh/sqlshield-fuzzy-sensitivity-v3/exp_fuzzy_group_sensitivity_v3.py \
    /kaggle/working/exp_fuzzy_group_sensitivity_v3.py

In [24]:
!python /kaggle/working/exp_fuzzy_group_sensitivity_v3.py \
    --base-dir /kaggle/working/sqlshield_corrected_validation \
    --cluster-only


SQLShield FUZZY / NEAR-DUPLICATE SENSITIVITY EXPERIMENT
Base dir:       /kaggle/working/sqlshield_corrected_validation
Output dir:     /kaggle/working/sqlshield_corrected_validation/fuzzy_sensitivity
Thresholds:     baseline 1.0 + [0.9, 0.8, 0.7]
Shingles:       5-char
MinHash perms:  256
LSH margin:     0.05
CodeBERT seed:  42
Cluster only:   True

Reconstructed corrected corpus: 56,621 rows
Exact normalized groups:        45,915
Class counts: benign=34,490 | SQLi=22,131

Preparing 45,915 exact normalized groups | 5-char shingles | num_perm=256
  MinHash prepared: 5,000/45,915 (0.0 min)
  MinHash prepared: 10,000/45,915 (0.1 min)
  MinHash prepared: 15,000/45,915 (0.1 min)
  MinHash prepared: 20,000/45,915 (0.1 min)
  MinHash prepared: 25,000/45,915 (0.2 min)
  MinHash prepared: 30,000/45,915 (0.2 min)
  MinHash prepared: 35,000/45,915 (0.2 min)
  MinHash prepared: 40,000/45,915 (0.2 min)
  MinHash prepared: 45,000/45,915 (0.3 min)
  MinHash prepared: 45,915/45,915 (0.3 min)

FUZZY C

In [25]:
!python /kaggle/working/exp_fuzzy_group_sensitivity_v3.py \
    --base-dir /kaggle/working/sqlshield_corrected_validation


SQLShield FUZZY / NEAR-DUPLICATE SENSITIVITY EXPERIMENT
Base dir:       /kaggle/working/sqlshield_corrected_validation
Output dir:     /kaggle/working/sqlshield_corrected_validation/fuzzy_sensitivity
Thresholds:     baseline 1.0 + [0.9, 0.8, 0.7]
Shingles:       5-char
MinHash perms:  256
LSH margin:     0.05
CodeBERT seed:  42
Cluster only:   False

Reconstructed corrected corpus: 56,621 rows
Exact normalized groups:        45,915
Class counts: benign=34,490 | SQLi=22,131

Preparing 45,915 exact normalized groups | 5-char shingles | num_perm=256
  MinHash prepared: 5,000/45,915 (0.0 min)
  MinHash prepared: 10,000/45,915 (0.1 min)
  MinHash prepared: 15,000/45,915 (0.1 min)
  MinHash prepared: 20,000/45,915 (0.1 min)
  MinHash prepared: 25,000/45,915 (0.2 min)
  MinHash prepared: 30,000/45,915 (0.2 min)
  MinHash prepared: 35,000/45,915 (0.2 min)
  MinHash prepared: 40,000/45,915 (0.3 min)
  MinHash prepared: 45,000/45,915 (0.3 min)
  MinHash prepared: 45,915/45,915 (0.3 min)

FUZZY 

In [26]:
import shutil

shutil.make_archive(
    "/kaggle/working/sqlshield_fuzzy_results",
    "zip",
    "/kaggle/working/sqlshield_corrected_validation/fuzzy_sensitivity"
)

print("/kaggle/working/sqlshield_fuzzy_results.zip")

/kaggle/working/sqlshield_fuzzy_results.zip


In [27]:
from pathlib import Path

p = Path("/kaggle/working/sqlshield_fuzzy_results.zip")
print(p.exists(), p.stat().st_size)

True 1397515595


In [29]:
from pathlib import Path
import pandas as pd

base = Path(
    "/kaggle/working/sqlshield_corrected_validation/fuzzy_sensitivity"
)

print("\n=== FUZZY SENSITIVITY RESULTS ===")
results = pd.read_csv(base / "fuzzy_sensitivity_results.csv")
print(results.to_string(index=False))

print("\n=== FUZZY CLUSTER AUDIT ===")
audit = pd.read_csv(base / "fuzzy_cluster_audit.csv")
print(audit.to_string(index=False))


=== FUZZY SENSITIVITY RESULTS ===
 threshold                                 grouping    model                model_id  seed  best_epoch  best_val_f1  accuracy  precision   recall       f1  roc_auc   pr_auc  balanced_accuracy      mcc   tn  fp  fn   tp  errors    n  elapsed_seconds  train_n  val_n  test_n  test_sqli  test_benign
       1.0  existing_exact_normalized_text_baseline CodeBERT microsoft/codebert-base    42         NaN          NaN  0.999469   0.999546 0.999093 0.999319 0.999971 0.999959           0.999401 0.998885 3449   1   2 2202       3 5654              NaN    45288   5679    5654       2204         3450
       0.9 fuzzy_5char_jaccard_connected_components CodeBERT microsoft/codebert-base    42         4.0     0.999776  0.999115   0.999546 0.998186 0.998865 0.999965 0.999949           0.998948 0.998141 3446   1   4 2201       5 5652      4030.732533    45291   5678    5652       2205         3447
       0.8 fuzzy_5char_jaccard_connected_components CodeBERT microsoft/cod